# 스마트 창고 출고 지연 예측 AI 경진대회 - Submission Package

**최고 Public LB: 9.802014278** (v9 불확실성 메타 스태킹)


## solution.md


# 스마트 창고 출고 지연 예측 AI 경진대회 - 솔루션

## 1. 문제 정의 및 접근 전략

### 1.1 문제 요약
- **과제**: 스마트 물류창고 운영 스냅샷(15분 단위, 시나리오당 25 타임스텝) → 향후 30분 평균 출고 지연 시간(분) 예측
- **데이터**: 정형 (train 250,000행 × 94컬럼, test 50,000행, 보조 layout_info 300행)
- **타깃**: `avg_delay_minutes_next_30m` (연속값 회귀)
- **평가**: MAE (낮을수록 우수) / Public 30% · Private 70%

### 1.2 접근 전략
1. **PB-style 피처 엔지니어링**: lag/rolling/비율/충전 압력 등 도메인 특화 피처 714개 구성
2. **시나리오 집계 피처**: 25 타임스텝 전체에 걸친 mean/std/max/min 통계 84개 추가 (→ 798개)
3. **이종 모델 다양성**: GBDT 3종 + TabNet + Transformer + MLP의 9개 모델군, 3-seed × 5-fold
4. **LGB 메타 스태킹**: OOF 예측 + 예측 불확실성(std/range)을 메타 피처로 활용
5. **GroupKFold**: scenario_id 기준으로 시나리오 단위 정보 누수 방지

---

## 2. 피처 엔지니어링 (798개)

### 2.1 피처 구성 요약

| 그룹 | 수 | 설명 |
|------|-----|------|
| strict base | 198개 | lag(1,2), rolling(3,5), expanding 통계, timestep, layout |
| attack lead/forward | 107개 | 미래 타임스텝 예측을 위한 ops lead/forward 피처 |
| PB-style lag/rolling/ratio | 409개 | 배터리 압력, 충전 대기 압력, 수요-로봇 비율, onset 피처 등 |
| **시나리오 집계** | **84개** | 25 타임스텝 전체 mean/std/max/min (21개 컬럼 × 4 통계) |

### 2.2 핵심 피처 아이디어

**PB-style 피처 (sections 1~12)**
- `charge_pressure_pb`: 충전 중인 로봇 비율 × 충전 큐 × 배터리 긴급도
- `battery_pressure_pb`: 저배터리 비율의 지수 함수 가중치
- `demand_mass_per_robot`: 주문 유입량 / 활성 로봇 수 (로봇당 부하)
- `congestion_x_lowbat`: 혼잡도 × 저배터리 비율 (이중 압박 지표)
- Onset 피처: 충전/큐가 처음 발생하는 타임스텝 위치

**시나리오 집계 피처 (section 13)** — 가장 큰 단독 개선 효과
```
시나리오 내 25 타임스텝의 mean/std/max/min → 
현재 타임스텝 값과 전체 시나리오 패턴을 동시에 학습 가능
```

---

## 3. 모델 구성

### 3.1 GBDT 모델 (checkpoints_v8/)

| 모델명 | objective | target 변환 | seeds |
|--------|-----------|-------------|-------|
| `lgb_mae_log` | LightGBM MAE | log1p | 42, 123, 2026 |
| `lgb_huber_log` | LightGBM Huber | log1p | 42, 123, 2026 |
| `cat_mae_log` | CatBoost MAE | log1p | 42, 123, 2026 |
| `lgb_mae_raw` | LightGBM MAE | raw | 42, 123, 2026 |
| `xgb_mae_raw` | XGBoost MAE | raw | 42, 123, 2026 |

- 피처: 798개 (v8b prefix)
- 검증: GroupKFold(5, groups=scenario_id)
- 체크포인트: `checkpoints_v8/v8b_{name}_seed{seed}.pkl`

### 3.2 TabNet 모델 (checkpoints_v8_tabnet/, checkpoints_strict/)

| 모델명 | 피처 | 비고 |
|--------|------|------|
| `v8_tabnet` | 714개 (attack 기반) | pytorch-tabnet |
| `strict2_tabnet` | 198개 (strict base) | 구 피처셋 |

### 3.3 Transformer v4 (checkpoints_v8_transformer/)

```
Input: (B, 25, 798)  ← 25 timestep sequence
        |
  Linear projection → d=256
        |
  Pre-Norm Transformer (L=6, heads=8, d_ff=1024)
        |
  Sequence output → (B, 25, 1)
        |
    Target per timestep
```

- **핵심 설정**: L1Loss (MAE loss), lr=2e-4, warmup=10 epoch, cosine LR, patience=35, epoch=250
- bfloat16 AMP, GroupKFold(5)
- 체크포인트: `checkpoints_v8_transformer/v8b_transformer_v4_seed{seed}.pkl`

### 3.4 MLP (checkpoints_v8_transformer/)

```
Input: (B, 798)  ← flat feature vector (row-level)
        |
  512 → BN+GELU+Drop → 512 → BN+GELU+Drop → 256 → BN+GELU+Drop → 1
```

- L1Loss, lr=3e-4, cosine warmup, patience=30, epoch=300
- 체크포인트: `checkpoints_v8_transformer/v8b_mlp_seed{seed}.pkl`

---

## 4. 스태킹 메타러너

### 4.1 메타 피처 구성 (11개)

| 피처 | 수 |
|------|-----|
| 9개 모델의 OOF 예측값 | 9개 |
| **예측 std** (모델 불확실성) | 1개 |
| **예측 range** (max-min) | 1개 |

불확실성 피처가 핵심 개선 기여:
- CV: 8.3961 → 8.3942
- Public LB: 9.821 → **9.802**

### 4.2 메타러너: LightGBM

```python
params = {
    "objective": "mae", "num_leaves": 15,
    "learning_rate": 0.05, "min_child_samples": 100,
    "subsample": 0.8, "colsample_bytree": 1.0, "seed": 42,
}
```

GroupKFold(5), early_stopping=50, num_boost_round=3000

---

## 5. 핵심 발견 및 실험 이력

### 5.1 핵심 발견

1. **시나리오 집계 피처** (section 13): GBDT blend CV 8.5426 → 8.4806 (+0.0620). 가장 큰 단독 개선
2. **예측 불확실성 메타 피처**: 메타러너가 모델 불일치 상황을 더 잘 처리 → LB 9.821 → 9.802
3. **LGB 스태킹 > blend > ridge**: 메타러너 복잡도와 성능이 비례하지 않음 (num_leaves=15 최적)
4. **1D CNN 부적합**: 25 timestep이 너무 짧아 로컬 컨볼루션 패턴 추출 불가 (CV ≥ 10.21)
5. **CV-LB 일관성**: GroupKFold CV가 LB 방향과 잘 일치 → CV 기반 의사결정 신뢰 가능
6. **메타 모델 수 주의**: 9→11개 증가 시 오히려 CV 악화 (과적합)

### 5.2 제출 이력

| # | 파일 | Public LB | CV | 핵심 |
|---|------|----------|----|------|
| 1 | v8_transformer_v4_blend_noclip | 9.9850 | 8.5372 | TF v4 블렌드 기준선 |
| 2 | v8_stacking_v1 | 9.8347 | 8.4230 | LGB 스태킹 8모델 도입 |
| 3 | v8_stacking_v2 | 9.8215 | 8.4050 | v8b transformer(798피처) 반영 |
| 4 | v8_stacking_v3 | 9.8254 | 8.3961 | MLP 추가 (9모델) |
| **5** | **v8_stacking_v9_unc** | **9.8020** | **8.3942** | **예측 std/range 메타 피처** |

---

## 6. 최종 결과

| 항목 | 값 |
|------|-----|
| **최고 Public LB** | **9.802014278** |
| **최고 CV** | **8.3942** |
| **제출 파일** | `v8_stacking_v9_unc_submission.csv` |
| **시작 대비 개선** | 10.2990 → 9.8020 (−0.497) |

---

## 7. 개발 환경 및 재현 방법

### 7.1 개발 환경

| 항목 | 값 |
|------|-----|
| OS | Ubuntu (WSL2) / Linux 6.6.87 |
| GPU | NVIDIA RTX 5060 Ti 16GB |
| Python | 3.11 |
| PyTorch | 2.x (bfloat16 AMP) |
| LightGBM | 4.x |
| XGBoost | 2.x |
| CatBoost | 1.x |
| pytorch-tabnet | 4.x |
| scikit-learn | 1.x |
| pandas | 2.x |
| numpy | 1.x / 2.x |

### 7.2 재현 순서

```bash
# 1. 환경 설정
pip install -r requirements.txt

# 2. 데이터 배치
# data/train.csv, data/test.csv, data/layout_info.csv, data/sample_submission.csv

# 3. GBDT 학습 (798 피처, v8b prefix)
python run_experiments_v8.py --ckpt_prefix v8b

# 4. Transformer v4 학습 (798 피처, L1Loss)
python run_v8_transformer.py --v4

# 5. MLP 학습 (798 피처, L1Loss)
python run_v8_mlp.py

# 6. TabNet 학습 (선택 — 체크포인트 포함)
# python run_v8_tabnet.py

# 7. Stacking (불확실성 메타 피처 포함)
python run_stacking.py --tag v9
# → submissions/v8_stacking_v9_submission.csv
```

### 7.3 파일 구조

```
smart_warehouse/
├── config.py                   # 하이퍼파라미터 중앙 관리 (경로, 컬럼, 모델 파라미터)
├── dataset.py                  # 데이터 로드 (사용하지 않는 경우 run_experiments_v8.py가 직접 로드)
├── run_experiments_v8.py       # GBDT 학습 (--ckpt_prefix v8b, 798 피처)
├── run_v8_transformer.py       # Transformer v4/MLP 학습 (--v4, --mlp)
├── run_v8_mlp.py               # MLP 전용 학습 스크립트
├── run_v8_tabnet.py            # TabNet 학습
├── run_stacking.py             # 스태킹 메타러너 (--tag v9)
├── run_experiments_v5.py       # 공용 유틸리티 (load_ckpt, optimize_blend 등)
├── run_experiments_strict.py   # strict2_tabnet 체크포인트 경로 참조용
├── data/
│   ├── train.csv
│   ├── test.csv
│   ├── layout_info.csv
│   └── sample_submission.csv
├── checkpoints_v8/             # GBDT v8b 체크포인트 (90개 파일)
├── checkpoints_v8_transformer/ # TF v4 + MLP 체크포인트 (21개 파일)
├── checkpoints_v8_tabnet/      # TabNet 체크포인트 (18개 파일)
├── checkpoints_strict/         # strict2_tabnet 체크포인트 (18개 파일)
└── submissions/
    └── v8_stacking_v9_unc_submission.csv  ← 최고 LB (9.8020)
```

---

## 8. 코드 설명 (1000자 제한용)

## PB-style 피처 엔지니어링
로봇 운영 지표(배터리·혼잡도·충전 큐 등)의 lag/rolling/비율 피처 714개에 시나리오 내 25개 타임스텝 집계(mean/std/max/min) 84개를 추가, 총 798개 피처를 구성했습니다. 시나리오 집계 피처가 GBDT blend CV 8.54 → 8.48 개선의 핵심이었습니다.

## 이종 모델 9종
LightGBM(MAE·Huber), XGBoost(MAE), CatBoost(MAE), TabNet, Transformer(Pre-Norm d=256 L=6 L1Loss), MLP(TabMLP L1Loss)를 3-seed × 5-fold GroupKFold로 학습하여 다양성을 확보했습니다.

## 불확실성 메타 스태킹
9개 OOF 예측값 외에 예측 std(모델 불확실성)와 range(max-min)를 메타 피처로 추가한 LGB 스태킹(num_leaves=15) 적용. 불확실성 피처 추가가 Public LB 9.821 → 9.802 개선의 핵심 기여입니다.

## GroupKFold
scenario_id 기준 GroupKFold(5)로 시나리오 단위 정보 누수를 방지. GBDT는 log1p 타깃 변환, Transformer/MLP는 L1Loss 직접 학습.

## 재현
Score 복원: run_experiments_v8.py → run_v8_transformer.py --v4 → run_v8_mlp.py → run_stacking.py --tag v9
폴더: ./data/(데이터), ./checkpoints_v8/(GBDT), ./checkpoints_v8_transformer/(DL), ./checkpoints_v8_tabnet/(TabNet), ./checkpoints_strict/(strict2_tabnet)


---
## 재현 가이드

### 환경 설정
```bash
pip install lightgbm xgboost catboost pytorch-tabnet scikit-learn pandas numpy torch
```

### 데이터 배치
```
data/train.csv
data/test.csv
data/layout_info.csv
data/sample_submission.csv
```

### 실행 순서 (Private Score 복원)
```bash
# 1. GBDT 학습 (798 피처, v8b prefix)
python run_experiments_v8.py --ckpt_prefix v8b

# 2. Transformer v4 학습 (798 피처, L1Loss, ~6시간)
python run_v8_transformer.py --v4

# 3. MLP 학습 (798 피처, L1Loss, ~2시간)
python run_v8_mlp.py

# 4. Stacking (불확실성 메타 피처 포함)
python run_stacking.py --tag v9
# → submissions/v8_stacking_v9_submission.csv
```

### 체크포인트 사용 시 (재학습 없이 바로 스태킹)
체크포인트가 이미 있으면 스크립트가 자동으로 캐시 로드합니다:
```bash
python run_stacking.py --tag v9
```

### 체크포인트 폴더
- `checkpoints_v8/` — GBDT v8b (90개 파일)
- `checkpoints_v8_transformer/` — Transformer v4 + MLP (21개 파일)
- `checkpoints_v8_tabnet/` — TabNet (18개 파일)
- `checkpoints_strict/` — strict2 TabNet (18개 파일)


---
## config.py


In [ ]:
# config.py
# ============================================================
# 아래 코드를 config.py 파일로 저장하세요

"""
File: config.py
"""


In [ ]:
"""v1 설정 — 스마트 창고 출고 지연 예측"""

import os

# ── 경로 ──────────────────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")
LOG_DIR = os.path.join(BASE_DIR, "logs")
SUBMISSION_DIR = os.path.join(BASE_DIR, "submissions")
EXPERIMENT_DIR = os.path.join(BASE_DIR, "experiments")

TRAIN_FILE = os.path.join(DATA_DIR, "train.csv")
TEST_FILE = os.path.join(DATA_DIR, "test.csv")
LAYOUT_FILE = os.path.join(DATA_DIR, "layout_info.csv")
SAMPLE_SUB_FILE = os.path.join(DATA_DIR, "sample_submission.csv")

# ── 컬럼 ──────────────────────────────────────────────
TARGET = "avg_delay_minutes_next_30m"
ID_COL = "ID"
GROUP_COL = "scenario_id"
LAYOUT_KEY = "layout_id"
ID_COLS = [ID_COL, LAYOUT_KEY, GROUP_COL]

# ── 피처 엔지니어링 ──────────────────────────────────
USE_LOG_TARGET = True

LAG_FEATURES = [
    "low_battery_ratio", "battery_mean", "robot_idle",
    "order_inflow_15m", "congestion_score", "robot_charging",
    "max_zone_density",
]
LAG_STEPS = [1, 2]
ROLLING_WINDOWS = [3, 5]

EXPANDING_FEATURES = LAG_FEATURES

SCENARIO_STAT_FEATURES = [
    "order_inflow_15m", "congestion_score", "low_battery_ratio",
    "battery_mean", "robot_utilization",
]

LAYOUT_TYPE_MAP = {"grid": 0, "hub_spoke": 1, "hybrid": 2, "narrow": 3}

# ── 검증 ──────────────────────────────────────────────
SEED = 42
N_FOLDS = 5
EARLY_STOPPING_ROUNDS = 100

# ── 모델 ──────────────────────────────────────────────
LGB_PARAMS = {
    "objective": "mae",
    "metric": "mae",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "num_leaves": 127,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 1.0,
    "min_child_samples": 50,
    "verbosity": -1,
    "random_state": SEED,
}

XGB_PARAMS = {
    "objective": "reg:absoluteerror",
    "eval_metric": "mae",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 1.0,
    "min_child_weight": 50,
    "tree_method": "hist",
    "random_state": SEED,
    "verbosity": 0,
}

CAT_PARAMS = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "iterations": 3000,
    "learning_rate": 0.03,
    "depth": 8,
    "l2_leaf_reg": 3.0,
    "subsample": 0.7,
    "random_seed": SEED,
    "verbose": 100,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
}

ENSEMBLE_WEIGHTS = [0.4, 0.3, 0.3]  # LGB, XGB, CAT

# ── Sanity check ──────────────────────────────────────
SANITY_N_FOLDS = 2
SANITY_N_ESTIMATORS = 200


---
## dataset.py


In [ ]:
# dataset.py
# ============================================================
# 아래 코드를 dataset.py 파일로 저장하세요

"""
File: dataset.py
"""


In [ ]:
"""데이터 로드 + 피처 엔지니어링 파이프라인"""

import pandas as pd
import numpy as np
import config as cfg


def load_raw_data():
    train = pd.read_csv(cfg.TRAIN_FILE)
    test = pd.read_csv(cfg.TEST_FILE)
    layout = pd.read_csv(cfg.LAYOUT_FILE)
    return train, test, layout


def merge_layout(df: pd.DataFrame, layout: pd.DataFrame) -> pd.DataFrame:
    layout = layout.copy()
    layout["layout_type_enc"] = layout["layout_type"].map(cfg.LAYOUT_TYPE_MAP)
    layout = layout.drop(columns=["layout_type"])
    df = df.merge(layout, on=cfg.LAYOUT_KEY, how="left")
    return df


def add_timestep(df: pd.DataFrame) -> pd.DataFrame:
    df["timestep"] = df.groupby(cfg.GROUP_COL).cumcount()
    return df


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    for feat in cfg.LAG_FEATURES:
        grp = df.groupby(cfg.GROUP_COL)[feat]
        for lag in cfg.LAG_STEPS:
            df[f"{feat}_lag{lag}"] = grp.shift(lag)
        df[f"{feat}_diff1"] = df[feat] - df[f"{feat}_lag1"]
    return df


def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    for feat in cfg.LAG_FEATURES:
        grp = df.groupby(cfg.GROUP_COL)[feat]
        for w in cfg.ROLLING_WINDOWS:
            shifted = grp.shift(1)
            rolling = shifted.groupby(df[cfg.GROUP_COL]).rolling(w, min_periods=1)
            df[f"{feat}_rmean{w}"] = rolling.mean().reset_index(level=0, drop=True)
            df[f"{feat}_rstd{w}"] = rolling.std().reset_index(level=0, drop=True)
    return df


def add_expanding_features(df: pd.DataFrame) -> pd.DataFrame:
    for feat in cfg.EXPANDING_FEATURES:
        shifted = df.groupby(cfg.GROUP_COL)[feat].shift(1)
        expanding = shifted.groupby(df[cfg.GROUP_COL]).expanding(min_periods=1)
        df[f"{feat}_exp_mean"] = expanding.mean().reset_index(level=0, drop=True)
        df[f"{feat}_exp_std"] = expanding.std().reset_index(level=0, drop=True)
    return df


def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    df["battery_x_congestion"] = df["low_battery_ratio"] * df["congestion_score"]
    df["inflow_x_utilization"] = df["order_inflow_15m"] * df["robot_utilization"]
    df["charging_x_demand"] = df["robot_charging"] * df["order_inflow_15m"]
    df["idle_x_inflow"] = df["robot_idle"] * df["order_inflow_15m"]
    df["battery_per_congestion"] = df["battery_mean"] / (df["congestion_score"] + 1)

    # layout 기반 비율
    rt = df["robot_total"].replace(0, np.nan)
    df["active_ratio"] = df["robot_active"] / rt
    df["charging_ratio"] = df["robot_charging"] / rt
    df["idle_ratio"] = df["robot_idle"] / rt

    cc = df["charger_count"].replace(0, np.nan)
    df["charger_utilization"] = df["robot_charging"] / cc
    df["orders_per_robot"] = df["order_inflow_15m"] / rt
    return df


def add_scenario_stats(df: pd.DataFrame) -> pd.DataFrame:
    for feat in cfg.SCENARIO_STAT_FEATURES:
        grp = df.groupby(cfg.GROUP_COL)[feat]
        sc_mean = grp.transform("mean")
        df[f"{feat}_sc_mean"] = sc_mean
        df[f"{feat}_sc_std"] = grp.transform("std")
        df[f"{feat}_sc_max"] = grp.transform("max")
        df[f"{feat}_dev_from_sc"] = df[feat] - sc_mean
    return df


def build_features(train: pd.DataFrame, test: pd.DataFrame, layout: pd.DataFrame):
    """메인 피처 빌드. (train_fe, test_fe, feature_cols) 반환."""
    train = train.copy()
    test = test.copy()

    train["_is_train"] = 1
    test["_is_train"] = 0
    if cfg.TARGET not in test.columns:
        test[cfg.TARGET] = np.nan

    df = pd.concat([train, test], ignore_index=True)

    df = merge_layout(df, layout)
    df = add_timestep(df)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    df = add_expanding_features(df)
    df = add_interaction_features(df)
    df = add_scenario_stats(df)

    # 피처 컬럼 결정: ID/메타/타깃/_is_train 제외
    exclude = set(cfg.ID_COLS + [cfg.TARGET, "_is_train"])
    feature_cols = [c for c in df.columns if c not in exclude]

    train_fe = df[df["_is_train"] == 1].reset_index(drop=True)
    test_fe = df[df["_is_train"] == 0].reset_index(drop=True)
    train_fe = train_fe.drop(columns=["_is_train"])
    test_fe = test_fe.drop(columns=["_is_train"])

    print(f"[dataset] 피처 수: {len(feature_cols)}")
    print(f"[dataset] train: {train_fe.shape}, test: {test_fe.shape}")
    return train_fe, test_fe, feature_cols


if __name__ == "__main__":
    train, test, layout = load_raw_data()
    train_fe, test_fe, feature_cols = build_features(train, test, layout)
    print(f"\n피처 목록 ({len(feature_cols)}):")
    for c in feature_cols:
        print(f"  {c}")
    print(f"\ntrain NaN 비율: {train_fe[feature_cols].isnull().mean().mean():.4f}")
    print(f"test  NaN 비율: {test_fe[feature_cols].isnull().mean().mean():.4f}")


---
## run_experiments_v5.py (공용 유틸리티)


In [ ]:
# run_experiments_v5.py
# ============================================================
# 아래 코드를 run_experiments_v5.py 파일로 저장하세요

"""
File: run_experiments_v5.py
"""


In [ ]:
"""
v5 실험 마스터 스크립트 — 6개 연속 실행 (Pseudo-labeling 완전 제거)
v5-1: Multi-Seed Baseline (Pseudo 제거)
v5-2: Adversarial Sample Weighting
v5-3: Target Transform Diversity
v5-4: High-Load Regime Features
v5-5: FT-Transformer
v5-6: Grand Ensemble + Distribution Calibration

규칙: test 데이터는 어떠한 형태로도 학습에 활용 불가
"""

import os, sys, time, argparse, traceback, warnings, pickle
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar, minimize
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import config as cfg
from dataset import load_raw_data, build_features

warnings.filterwarnings("ignore")

os.makedirs(cfg.LOG_DIR, exist_ok=True)
os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)

CKPT_DIR = os.path.join(os.path.dirname(__file__), "checkpoints_v5")  # main() 에서 sanity 시 변경
os.makedirs(CKPT_DIR, exist_ok=True)

# ── 전역 상태 ──────────────────────────────────────────────────────
EXPD_LB = 10.2428
RESULTS = {}
V5_CACHE = {}  # 실험 간 캐시 공유


def log(msg):
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


# ── Checkpoint 유틸 ────────────────────────────────────────────────
def save_ckpt(name, data):
    path = os.path.join(CKPT_DIR, f"{name}.pkl")
    with open(path, "wb") as f:
        pickle.dump(data, f)
    log(f"  [ckpt] 저장: {name}")


def load_ckpt(name):
    path = os.path.join(CKPT_DIR, f"{name}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        log(f"  [ckpt] 로드: {name}")
        return data
    return None


# ── Target Transform ──────────────────────────────────────────────
def transform_target(y_raw, transform="log1p"):
    if transform == "log1p":
        return np.log1p(y_raw)
    elif transform == "sqrt":
        return np.sqrt(np.clip(y_raw, 0, None))
    return y_raw.copy()


def inverse_transform(preds, transform="log1p"):
    if transform == "log1p":
        preds = np.expm1(preds)
    elif transform == "sqrt":
        preds = np.clip(preds, 0, None) ** 2
    return np.clip(preds, 0, None)


# ── 학습 유틸 ─────────────────────────────────────────────────────
def train_lgb(X_tr, y_tr, X_val, y_val, params, early_stop, sample_weight=None):
    m = LGBMRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          sample_weight=sample_weight,
          callbacks=[lgb.early_stopping(early_stop), lgb.log_evaluation(200)])
    return m


def train_xgb(X_tr, y_tr, X_val, y_val, params, early_stop, sample_weight=None):
    p = params.copy()
    p["early_stopping_rounds"] = early_stop
    m = XGBRegressor(**p)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          sample_weight=sample_weight, verbose=200)
    return m


def train_cat(X_tr, y_tr, X_val, y_val, params, sample_weight=None):
    m = CatBoostRegressor(**params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), sample_weight=sample_weight)
    return m


def save_submission(test_fe, preds, name):
    sample_sub = pd.read_csv(cfg.SAMPLE_SUB_FILE)
    sub = pd.DataFrame({cfg.ID_COL: test_fe[cfg.ID_COL], cfg.TARGET: preds})
    assert len(sub) == len(sample_sub), f"행 수 불일치: {len(sub)} vs {len(sample_sub)}"
    assert sub[cfg.TARGET].isna().sum() == 0, "NaN 존재"
    assert (sub[cfg.TARGET] >= 0).all(), "음수 존재"
    path = os.path.join(cfg.SUBMISSION_DIR, f"{name}_submission.csv")
    sub.to_csv(path, index=False)
    log(f"  저장: {path}  mean={preds.mean():.2f} std={preds.std():.2f} max={preds.max():.2f}")
    return path


# ── Blend 최적화 ──────────────────────────────────────────────────
def optimize_blend_2way(oof_a, oof_b, y_raw):
    def mae_fn(alpha):
        return mean_absolute_error(y_raw, alpha * oof_a + (1 - alpha) * oof_b)
    result = minimize_scalar(mae_fn, bounds=(0.1, 0.95), method="bounded")
    best_a, best_mae = result.x, result.fun
    log(f"  2-way blend -> A:{best_a:.3f} / B:{1 - best_a:.3f}  CV MAE: {best_mae:.4f}")
    return best_a, best_mae


def optimize_blend_multi(oof_list, y_raw, names=None):
    n = len(oof_list)
    if names is None:
        names = [f"M{i}" for i in range(n)]

    def obj(w):
        w_abs = np.abs(w)
        w_norm = w_abs / w_abs.sum()
        pred = sum(w_norm[i] * oof_list[i] for i in range(n))
        return mean_absolute_error(y_raw, pred)

    x0 = np.ones(n) / n
    result = minimize(obj, x0, method="Nelder-Mead",
                      options={"maxiter": 10000, "xatol": 1e-6, "fatol": 1e-6})
    w = np.abs(result.x)
    w = w / w.sum()
    log(f"  {n}-way blend CV MAE: {result.fun:.4f}")
    for i, name in enumerate(names):
        log(f"    {name}: {w[i]:.3f}")
    return w, result.fun


# ── GBDT CV (pseudo 제거, transform 지원) ─────────────────────────
def run_gbdt_cv(X, y_raw, groups, X_test,
                transform="log1p",
                lgb_params=None, xgb_params=None, cat_params=None,
                weights=None, n_folds=5, early_stop=100,
                ckpt_name=None):
    if ckpt_name:
        cached = load_ckpt(ckpt_name)
        if cached is not None:
            return cached

    lgb_p = (lgb_params or cfg.LGB_PARAMS).copy()
    xgb_p = (xgb_params or cfg.XGB_PARAMS).copy()
    cat_p = (cat_params or cfg.CAT_PARAMS).copy()
    xgb_p["early_stopping_rounds"] = early_stop
    cat_p["early_stopping_rounds"] = early_stop

    y = transform_target(y_raw, transform)

    gkf = GroupKFold(n_splits=n_folds)
    oof_lgb = np.zeros(len(X));  test_lgb = np.zeros(len(X_test))
    oof_xgb = np.zeros(len(X));  test_xgb = np.zeros(len(X_test))
    oof_cat = np.zeros(len(X));  test_cat = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        fold_ckpt = f"{ckpt_name}_fold{fold}" if ckpt_name else None
        if fold_ckpt:
            fc = load_ckpt(fold_ckpt)
            if fc is not None:
                oof_lgb[val_idx] = fc["oof_lgb"]; test_lgb += fc["test_lgb"] / n_folds
                oof_xgb[val_idx] = fc["oof_xgb"]; test_xgb += fc["test_xgb"] / n_folds
                oof_cat[val_idx] = fc["oof_cat"]; test_cat += fc["test_cat"] / n_folds
                print(f"  Fold {fold+1} ENS MAE: {fc['ens_mae']:.4f}  (캐시)")
                continue

        ft0 = time.time()
        print(f"\n  Fold {fold+1}/{n_folds}  (train={len(tr_idx)}, val={len(val_idx)})")

        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        y_val_raw = y_raw[val_idx]
        sw = weights[tr_idx] if weights is not None else None

        m_lgb = train_lgb(X_tr, y_tr, X_val, y_val, lgb_p, early_stop, sw)
        oof_lgb[val_idx] = inverse_transform(m_lgb.predict(X_val), transform)
        fold_test_lgb = inverse_transform(m_lgb.predict(X_test), transform)
        test_lgb += fold_test_lgb / n_folds

        m_xgb = train_xgb(X_tr, y_tr, X_val, y_val, xgb_p, early_stop, sw)
        oof_xgb[val_idx] = inverse_transform(m_xgb.predict(X_val), transform)
        fold_test_xgb = inverse_transform(m_xgb.predict(X_test), transform)
        test_xgb += fold_test_xgb / n_folds

        m_cat = train_cat(X_tr, y_tr, X_val, y_val, cat_p, sw)
        oof_cat[val_idx] = inverse_transform(m_cat.predict(X_val), transform)
        fold_test_cat = inverse_transform(m_cat.predict(X_test), transform)
        test_cat += fold_test_cat / n_folds

        w = cfg.ENSEMBLE_WEIGHTS
        p_ens = w[0]*oof_lgb[val_idx] + w[1]*oof_xgb[val_idx] + w[2]*oof_cat[val_idx]
        ens_mae = mean_absolute_error(y_val_raw, p_ens)
        print(f"  Fold {fold+1} ENS MAE: {ens_mae:.4f}  ({time.time()-ft0:.0f}s)")

        if fold_ckpt:
            save_ckpt(fold_ckpt, {
                "oof_lgb": oof_lgb[val_idx], "test_lgb": fold_test_lgb,
                "oof_xgb": oof_xgb[val_idx], "test_xgb": fold_test_xgb,
                "oof_cat": oof_cat[val_idx], "test_cat": fold_test_cat,
                "ens_mae": ens_mae,
            })

    w = cfg.ENSEMBLE_WEIGHTS
    oof_ens = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cat
    test_ens = w[0]*test_lgb + w[1]*test_xgb + w[2]*test_cat
    cv_mae = mean_absolute_error(y_raw, oof_ens)

    result = (cv_mae, test_ens, oof_ens)
    if ckpt_name:
        save_ckpt(ckpt_name, result)
    return result


def run_gbdt_multiseed(X, y_raw, groups, X_test, seeds=(42, 123, 2026),
                       ckpt_prefix="gbdt", **kwargs):
    all_oof, all_test = [], []
    for seed in seeds:
        lgb_p = (kwargs.get("lgb_params") or cfg.LGB_PARAMS).copy()
        xgb_p = (kwargs.get("xgb_params") or cfg.XGB_PARAMS).copy()
        cat_p = (kwargs.get("cat_params") or cfg.CAT_PARAMS).copy()
        lgb_p["random_state"] = seed
        xgb_p["random_state"] = seed
        cat_p["random_seed"] = seed

        kw = {k: v for k, v in kwargs.items() if k not in ("lgb_params", "xgb_params", "cat_params")}
        cv, test, oof = run_gbdt_cv(
            X, y_raw, groups, X_test,
            lgb_params=lgb_p, xgb_params=xgb_p, cat_params=cat_p,
            ckpt_name=f"{ckpt_prefix}_seed{seed}", **kw)
        all_oof.append(oof)
        all_test.append(test)
        log(f"  GBDT seed={seed} CV: {cv:.4f}")

    oof_avg = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_avg = mean_absolute_error(y_raw, oof_avg)
    log(f"  GBDT {len(seeds)}-seed avg CV: {cv_avg:.4f}")
    return cv_avg, test_avg, oof_avg


# ── TabNet CV (pseudo 제거) ────────────────────────────────────────
def run_tabnet_cv(X_arr, y_raw, groups, X_test_arr,
                  n_d=32, n_a=32, n_steps=5, patience=20, batch_size=4096,
                  n_folds=5, base_seed=42, max_epochs=200,
                  ckpt_name=None):
    if ckpt_name:
        cached = load_ckpt(ckpt_name)
        if cached is not None:
            return cached

    from pytorch_tabnet.tab_model import TabNetRegressor
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"  TabNet -- device:{device}  n_d={n_d}  n_steps={n_steps}  seed={base_seed}")

    y_log = np.log1p(y_raw).astype(np.float32).reshape(-1, 1)

    gkf = GroupKFold(n_splits=n_folds)
    oof_tab = np.zeros(len(X_arr))
    test_tab = np.zeros(len(X_test_arr))

    fold_ckpt_prefix = f"{ckpt_name}_fold" if ckpt_name else None
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_arr, groups=groups)):
        if fold_ckpt_prefix:
            fold_cached = load_ckpt(f"{fold_ckpt_prefix}_{fold}")
            if fold_cached is not None:
                oof_tab[val_idx] = fold_cached["oof"]
                test_tab += fold_cached["test"] / n_folds
                log(f"  Fold {fold+1} TabNet MAE: {fold_cached['mae']:.4f}  (캐시)")
                continue

        ft0 = time.time()
        X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
        y_tr, y_val = y_log[tr_idx], y_log[val_idx]
        y_val_raw = y_raw[val_idx]

        import torch
        tab = TabNetRegressor(
            n_d=n_d, n_a=n_a, n_steps=n_steps,
            gamma=1.5, n_independent=2, n_shared=2,
            lambda_sparse=1e-4,
            optimizer_params=dict(lr=2e-3, weight_decay=1e-5),
            scheduler_params=dict(step_size=10, gamma=0.9),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            mask_type="entmax",
            device_name=device, verbose=0,
            seed=base_seed + fold,
        )
        tab.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric=["mae"],
                max_epochs=max_epochs, patience=patience,
                batch_size=batch_size, virtual_batch_size=256)

        p = inverse_transform(tab.predict(X_val).flatten(), "log1p")
        test_contrib = inverse_transform(tab.predict(X_test_arr).flatten(), "log1p")
        oof_tab[val_idx] = p
        test_tab += test_contrib / n_folds
        fold_mae = mean_absolute_error(y_val_raw, p)
        log(f"  Fold {fold+1} TabNet MAE: {fold_mae:.4f}  ({time.time()-ft0:.0f}s)")

        if fold_ckpt_prefix:
            save_ckpt(f"{fold_ckpt_prefix}_{fold}", {
                "oof": p, "test": test_contrib, "mae": fold_mae,
            })

    cv_tab = mean_absolute_error(y_raw, oof_tab)
    log(f"  TabNet CV MAE: {cv_tab:.4f}")
    result = (cv_tab, test_tab, oof_tab)
    if ckpt_name:
        save_ckpt(ckpt_name, result)
    return result


def run_tabnet_multiseed(X_arr, y_raw, groups, X_test_arr, seeds=(42, 123, 2026),
                         ckpt_prefix="tabnet", **kwargs):
    all_oof, all_test = [], []
    for seed in seeds:
        cv, test, oof = run_tabnet_cv(
            X_arr, y_raw, groups, X_test_arr,
            base_seed=seed, ckpt_name=f"{ckpt_prefix}_seed{seed}", **kwargs)
        all_oof.append(oof)
        all_test.append(test)

    oof_avg = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_avg = mean_absolute_error(y_raw, oof_avg)
    log(f"  TabNet {len(seeds)}-seed avg CV: {cv_avg:.4f}")
    return cv_avg, test_avg, oof_avg


# ── Adversarial Weighting ─────────────────────────────────────────
def compute_adversarial_weights(X_train, X_test, feature_cols, beta=1.0):
    X_all = pd.concat([X_train[feature_cols], X_test[feature_cols]], ignore_index=True)
    y_adv = np.array([0]*len(X_train) + [1]*len(X_test))
    clf = LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        verbosity=-1, random_state=42,
    )
    clf.fit(X_all, y_adv)
    p_test = clf.predict_proba(X_train[feature_cols])[:, 1]
    auc = np.mean((p_test > 0.5).astype(int))
    log(f"  Adversarial AUC proxy: {auc:.3f}  (train이 test처럼 보이는 비율)")

    weights = 1.0 + beta * p_test
    log(f"  beta={beta}  weights: mean={weights.mean():.3f} min={weights.min():.3f} max={weights.max():.3f}")
    return weights, p_test


def tune_adversarial_beta(X, y_raw, groups, p_test, betas=(0.5, 1.0, 2.0, 3.0), n_folds=5):
    best_beta, best_cv = 0.0, float("inf")
    lgb_p = cfg.LGB_PARAMS.copy()
    lgb_p["n_estimators"] = 500

    for beta in betas:
        weights = 1.0 + beta * p_test
        y = np.log1p(y_raw)
        gkf = GroupKFold(n_splits=n_folds)
        fold_maes = []
        for _, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
            m = LGBMRegressor(**lgb_p)
            m.fit(X.iloc[tr_idx], y[tr_idx],
                  eval_set=[(X.iloc[val_idx], y[val_idx])],
                  sample_weight=weights[tr_idx],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(9999)])
            preds = inverse_transform(m.predict(X.iloc[val_idx]), "log1p")
            fold_maes.append(mean_absolute_error(y_raw[val_idx], preds))
        avg = np.mean(fold_maes)
        log(f"  beta={beta:.1f}  avg CV: {avg:.4f}  worst: {max(fold_maes):.4f}")
        if avg < best_cv:
            best_cv, best_beta = avg, beta

    log(f"  최적 beta: {best_beta}")
    return best_beta


# ── High-Load Features (v5-4) ─────────────────────────────────────
def add_highload_features(train_fe, test_fe, feature_cols):
    """고부하 특화 피처 추가. 임계값은 train에서만 산출."""
    train_fe = train_fe.copy()
    test_fe = test_fe.copy()

    # Train 기준 임계값
    thresholds = {
        "order_p75": np.nanpercentile(train_fe["order_inflow_15m"], 75),
        "battery_p90": np.nanpercentile(train_fe["low_battery_ratio"], 90),
        "congestion_p90": np.nanpercentile(train_fe["congestion_score"], 90),
    }
    log(f"  임계값: order_P75={thresholds['order_p75']:.2f}  "
        f"battery_P90={thresholds['battery_p90']:.4f}  congestion_P90={thresholds['congestion_p90']:.2f}")

    for df in [train_fe, test_fe]:
        rt = df["robot_total"].replace(0, np.nan)
        cc = df["charger_count"].replace(0, np.nan)

        # 1. 포화 지표
        df["order_surge"] = (df["order_inflow_15m"] > thresholds["order_p75"]).astype(np.float32)
        df["battery_critical"] = (df["low_battery_ratio"] > thresholds["battery_p90"]).astype(np.float32)
        df["congestion_critical"] = (df["congestion_score"] > thresholds["congestion_p90"]).astype(np.float32)
        df["robot_capacity_used"] = (df["robot_active"] + df["robot_charging"]) / rt

        # 2. 스트레스 상호작용
        df["stress_index"] = df["order_inflow_15m"] * df["congestion_score"] * df["low_battery_ratio"]
        df["bottleneck_score"] = df["max_zone_density"] * (1 - df["robot_idle"] / rt)
        df["recovery_pressure"] = df["charge_queue_length"] * df["avg_charge_wait"] / (cc + 1)
        df["demand_supply_gap"] = df["order_inflow_15m"] - df["robot_active"]
        df["cascade_risk"] = df["fault_count_15m"] * df["congestion_score"] * df["blocked_path_15m"]

        # 3. 레이아웃 용량비
        df["pack_station_per_robot"] = df["pack_station_count"] / rt
        df["charger_per_robot"] = df["charger_count"] / rt

    # 4. 시간적 스트레스 (시나리오 내 변화)
    for df in [train_fe, test_fe]:
        grp = df.groupby(cfg.GROUP_COL)["stress_index"]
        df["stress_acceleration"] = df["stress_index"] - grp.shift(1)
        shifted = grp.shift(1)
        df["sustained_stress"] = shifted.groupby(df[cfg.GROUP_COL]).rolling(
            3, min_periods=1).mean().reset_index(level=0, drop=True)
        sc_max = grp.transform("max").replace(0, np.nan)
        df["peak_stress_ratio"] = df["stress_index"] / sc_max

    new_cols = [
        "order_surge", "battery_critical", "congestion_critical", "robot_capacity_used",
        "stress_index", "bottleneck_score", "recovery_pressure", "demand_supply_gap", "cascade_risk",
        "pack_station_per_robot", "charger_per_robot",
        "stress_acceleration", "sustained_stress", "peak_stress_ratio",
    ]
    extended_cols = feature_cols + new_cols
    log(f"  피처: {len(feature_cols)} -> {len(extended_cols)} (+{len(new_cols)})")
    return train_fe, test_fe, extended_cols


def adversarial_check_features(X_train, X_test, feature_cols, new_cols):
    """신규 피처가 train-test 구분력을 높이는지 확인."""
    X_all_base = pd.concat([X_train[feature_cols], X_test[feature_cols]], ignore_index=True)
    extended = feature_cols + new_cols
    X_all_ext = pd.concat([X_train[extended], X_test[extended]], ignore_index=True)
    y_adv = np.array([0]*len(X_train) + [1]*len(X_test))

    clf_base = LGBMClassifier(n_estimators=300, max_depth=5, verbosity=-1, random_state=42)
    clf_base.fit(X_all_base, y_adv)
    auc_base = clf_base.score(X_all_base, y_adv)

    clf_ext = LGBMClassifier(n_estimators=300, max_depth=5, verbosity=-1, random_state=42)
    clf_ext.fit(X_all_ext, y_adv)
    auc_ext = clf_ext.score(X_all_ext, y_adv)

    log(f"  Adversarial check: base={auc_base:.4f} -> extended={auc_ext:.4f} (diff={auc_ext-auc_base:+.4f})")

    # 개별 신규 피처 중요도 확인
    imp = pd.Series(clf_ext.feature_importances_, index=extended).sort_values(ascending=False)
    problematic = [c for c in new_cols if imp.get(c, 0) > imp.median() * 3]
    if problematic:
        log(f"  경고: 구분력 높은 신규 피처: {problematic}")
    return problematic, auc_ext - auc_base


# ── FT-Transformer ────────────────────────────────────────────────
def build_ft_transformer(n_features, d_model=64, n_heads=4, n_layers=2,
                         d_ff=128, dropout=0.2):
    import torch
    import torch.nn as nn

    class FTTransformer(nn.Module):
        def __init__(self):
            super().__init__()
            self.tokenizers = nn.ModuleList([
                nn.Linear(1, d_model) for _ in range(n_features)
            ])
            self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                dropout=dropout, batch_first=True, norm_first=True,
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
            self.norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, 1)

        def forward(self, x):
            tokens = torch.stack([tok(x[:, i:i+1]) for i, tok in enumerate(self.tokenizers)], dim=1)
            cls = self.cls_token.expand(x.size(0), -1, -1)
            tokens = torch.cat([cls, tokens], dim=1)
            tokens = self.transformer(tokens)
            return self.head(self.norm(tokens[:, 0])).squeeze(-1)

    return FTTransformer()


def run_ft_transformer_cv(X_arr, y_raw, groups, X_test_arr,
                          n_folds=5, base_seed=42, max_epochs=100, patience=15,
                          batch_size=512, d_model=64, n_heads=4, n_layers=2,
                          ckpt_name=None):
    if ckpt_name:
        cached = load_ckpt(ckpt_name)
        if cached is not None:
            return cached

    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    # GPU 메모리 정리
    torch.cuda.empty_cache()
    import gc; gc.collect()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_features = X_arr.shape[1]
    y_log = np.log1p(y_raw).astype(np.float32)

    gkf = GroupKFold(n_splits=n_folds)
    oof_ft = np.zeros(len(X_arr))
    test_ft = np.zeros(len(X_test_arr))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_arr, groups=groups)):
        fold_ckpt = f"{ckpt_name}_fold_{fold}" if ckpt_name else None
        if fold_ckpt:
            fc = load_ckpt(fold_ckpt)
            if fc is not None:
                oof_ft[val_idx] = fc["oof"]
                test_ft += fc["test"] / n_folds
                log(f"  Fold {fold+1} FT MAE: {fc['mae']:.4f}  (캐시)")
                continue

        ft0 = time.time()
        torch.manual_seed(base_seed + fold)
        np.random.seed(base_seed + fold)

        torch.cuda.empty_cache()
        X_tr_t = torch.FloatTensor(X_arr[tr_idx]).to(device)
        y_tr_t = torch.FloatTensor(y_log[tr_idx]).to(device)
        X_val_np = X_arr[val_idx]
        X_val_t = torch.FloatTensor(X_val_np).to(device)

        train_ds = TensorDataset(X_tr_t, y_tr_t)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

        model = build_ft_transformer(n_features, d_model=d_model, n_heads=n_heads,
                                     n_layers=n_layers).to(device).to(torch.bfloat16)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
        criterion = nn.MSELoss()

        def _batch_predict_np(model_, X_np, bs=batch_size):
            """CPU numpy → GPU batch → CPU numpy"""
            preds = []
            for i in range(0, len(X_np), bs):
                xb = torch.FloatTensor(X_np[i:i+bs]).to(device).to(torch.bfloat16)
                preds.append(model_(xb).float().cpu().numpy())
            return np.concatenate(preds)

        best_mae, best_state, wait = float("inf"), None, 0
        for epoch in range(max_epochs):
            model.train()
            for xb, yb in train_dl:
                pred = model(xb.to(torch.bfloat16))
                loss = criterion(pred.float(), yb)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                val_pred = _batch_predict_np(model, X_val_np)
            val_mae = mean_absolute_error(y_raw[val_idx], inverse_transform(val_pred, "log1p"))

            if val_mae < best_mae:
                best_mae = val_mae
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            p = inverse_transform(_batch_predict_np(model, X_val_np), "log1p")
            test_contrib = inverse_transform(_batch_predict_np(model, X_test_arr), "log1p")

        oof_ft[val_idx] = p
        test_ft += test_contrib / n_folds
        fold_mae = mean_absolute_error(y_raw[val_idx], p)
        log(f"  Fold {fold+1} FT MAE: {fold_mae:.4f}  ep={epoch+1} ({time.time()-ft0:.0f}s)")

        if fold_ckpt:
            save_ckpt(fold_ckpt, {"oof": p, "test": test_contrib, "mae": fold_mae})

        # GPU 메모리 정리
        del model, X_tr_t, y_tr_t
        torch.cuda.empty_cache()

    cv_ft = mean_absolute_error(y_raw, oof_ft)
    log(f"  FT-Transformer CV MAE: {cv_ft:.4f}")
    result = (cv_ft, test_ft, oof_ft)
    if ckpt_name:
        save_ckpt(ckpt_name, result)
    return result


def run_ft_multiseed(X_arr, y_raw, groups, X_test_arr, seeds=(42, 123, 2026),
                     ckpt_prefix="ft", **kwargs):
    all_oof, all_test = [], []
    for seed in seeds:
        cv, test, oof = run_ft_transformer_cv(
            X_arr, y_raw, groups, X_test_arr,
            base_seed=seed, ckpt_name=f"{ckpt_prefix}_seed{seed}", **kwargs)
        all_oof.append(oof)
        all_test.append(test)

    oof_avg = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_avg = mean_absolute_error(y_raw, oof_avg)
    log(f"  FT {len(seeds)}-seed avg CV: {cv_avg:.4f}")
    return cv_avg, test_avg, oof_avg


# ── P99 Clipping ──────────────────────────────────────────────────
def apply_p99_clipping(preds, factor=1.1):
    p99 = np.percentile(preds, 99)
    clip_val = p99 * factor
    n_clip = (preds > clip_val).sum()
    clipped = np.clip(preds, 0, clip_val)
    log(f"  P99 clipping: p99={p99:.2f} clip={clip_val:.2f} n_clip={n_clip}")
    return clipped


# ══════════════════════════════════════════════════════════════════
# ── v5-1: Multi-Seed Baseline ────────────────────────────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_1(data):
    log("=" * 60)
    log("v5-1: Multi-Seed GBDT + Multi-Seed TabNet Baseline (NO Pseudo)")
    log("=" * 60)
    t0 = time.time()

    train_fe, test_fe = data["train_fe"], data["test_fe"]
    feature_cols = data["feature_cols"]
    sanity = data.get("sanity", False)
    n_folds = 2 if sanity else 5
    max_ep = 10 if sanity else 200
    seeds = [42] if sanity else [42, 123, 2026]

    X = train_fe[feature_cols]
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values
    X_test = test_fe[feature_cols]

    # Multi-seed GBDT
    log("  [1/3] Multi-seed GBDT...")
    cv_gbdt, test_gbdt, oof_gbdt = run_gbdt_multiseed(
        X, y_raw, groups, X_test, seeds=seeds,
        ckpt_prefix="v5_1_gbdt", n_folds=n_folds)

    # Multi-seed TabNet
    log("  [2/3] Multi-seed TabNet...")
    X_arr = X.fillna(0).values.astype(np.float32)
    Xt_arr = X_test.fillna(0).values.astype(np.float32)
    cv_tab, test_tab, oof_tab = run_tabnet_multiseed(
        X_arr, y_raw, groups, Xt_arr, seeds=seeds,
        ckpt_prefix="v5_1_tabnet", n_folds=n_folds, max_epochs=max_ep)

    # Blend + Post-processing
    log("  [3/3] Blend + 후처리...")
    best_a, cv_ens = optimize_blend_2way(oof_gbdt, oof_tab, y_raw)
    test_ens = best_a * test_gbdt + (1 - best_a) * test_tab
    test_pp = apply_p99_clipping(test_ens)
    save_submission(test_fe, test_pp, "v5-1")

    # 캐시 저장
    V5_CACHE["v1_oof_gbdt"] = oof_gbdt
    V5_CACHE["v1_test_gbdt"] = test_gbdt
    V5_CACHE["v1_oof_tab"] = oof_tab
    V5_CACHE["v1_test_tab"] = test_tab

    log(f"v5-1 완료 -- GBDT:{cv_gbdt:.4f} | Tab:{cv_tab:.4f} | ENS:{cv_ens:.4f} ({(time.time()-t0)/60:.1f}분)")
    return cv_ens


# ══════════════════════════════════════════════════════════════════
# ── v5-2: Adversarial Sample Weighting ───────────────────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_2(data):
    log("=" * 60)
    log("v5-2: Adversarial Sample Weighting")
    log("=" * 60)
    t0 = time.time()

    train_fe, test_fe = data["train_fe"], data["test_fe"]
    feature_cols = data["feature_cols"]
    sanity = data.get("sanity", False)
    n_folds = 2 if sanity else 5
    max_ep = 10 if sanity else 200
    seeds = [42] if sanity else [42, 123, 2026]

    X = train_fe[feature_cols]
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values
    X_test = test_fe[feature_cols]

    # 1) Adversarial weighting
    log("  [1/4] Adversarial weight 계산...")
    _, p_test = compute_adversarial_weights(train_fe, test_fe, feature_cols)

    # 2) Beta 튜닝
    log("  [2/4] Beta 튜닝...")
    betas = [0.5, 1.0] if sanity else [0.5, 1.0, 2.0, 3.0]
    best_beta = tune_adversarial_beta(X, y_raw, groups, p_test, betas=betas, n_folds=n_folds)
    weights = 1.0 + best_beta * p_test

    # 3) Multi-seed GBDT with weights
    log("  [3/4] Weighted multi-seed GBDT...")
    cv_gbdt, test_gbdt, oof_gbdt = run_gbdt_multiseed(
        X, y_raw, groups, X_test, seeds=seeds,
        weights=weights, ckpt_prefix="v5_2_gbdt", n_folds=n_folds)

    # 4) TabNet (oversampling으로 가중치 반영)
    log("  [4/4] Weighted TabNet...")
    X_arr = X.fillna(0).values.astype(np.float32)
    Xt_arr = X_test.fillna(0).values.astype(np.float32)
    # TabNet은 v5-1 캐시 재사용 (sample_weight 미지원이므로 동일)
    if "v1_oof_tab" in V5_CACHE:
        log("  TabNet: v5-1 캐시 재사용")
        oof_tab = V5_CACHE["v1_oof_tab"]
        test_tab = V5_CACHE["v1_test_tab"]
    else:
        _, test_tab, oof_tab = run_tabnet_multiseed(
            X_arr, y_raw, groups, Xt_arr, seeds=seeds,
            ckpt_prefix="v5_2_tabnet", n_folds=n_folds, max_epochs=max_ep)

    best_a, cv_ens = optimize_blend_2way(oof_gbdt, oof_tab, y_raw)
    test_ens = best_a * test_gbdt + (1 - best_a) * test_tab
    test_pp = apply_p99_clipping(test_ens)
    save_submission(test_fe, test_pp, "v5-2")

    V5_CACHE["v2_oof_gbdt"] = oof_gbdt
    V5_CACHE["v2_test_gbdt"] = test_gbdt
    V5_CACHE["adv_weights"] = weights

    log(f"v5-2 완료 -- GBDT:{cv_gbdt:.4f} | ENS:{cv_ens:.4f} beta={best_beta} ({(time.time()-t0)/60:.1f}분)")
    return cv_ens


# ══════════════════════════════════════════════════════════════════
# ── v5-3: Target Transform Diversity ─────────────────────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_3(data):
    log("=" * 60)
    log("v5-3: Target Transform Diversity (log1p + sqrt + Huber)")
    log("=" * 60)
    t0 = time.time()

    train_fe, test_fe = data["train_fe"], data["test_fe"]
    feature_cols = data["feature_cols"]
    sanity = data.get("sanity", False)
    n_folds = 2 if sanity else 5

    X = train_fe[feature_cols]
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values
    X_test = test_fe[feature_cols]

    # Pipeline A: log1p + MAE (v5-1 캐시 재사용)
    log("  [1/3] Pipeline A: log1p + MAE (v5-1 캐시)...")
    if "v1_oof_gbdt" in V5_CACHE:
        oof_a = V5_CACHE["v1_oof_gbdt"]
        test_a = V5_CACHE["v1_test_gbdt"]
        cv_a = mean_absolute_error(y_raw, oof_a)
        log(f"  Pipeline A CV: {cv_a:.4f} (캐시)")
    else:
        cv_a, test_a, oof_a = run_gbdt_cv(
            X, y_raw, groups, X_test, transform="log1p",
            ckpt_name="v5_3_pipe_a", n_folds=n_folds)

    # Pipeline B: sqrt + MAE
    log("  [2/3] Pipeline B: sqrt + MAE...")
    cv_b, test_b, oof_b = run_gbdt_cv(
        X, y_raw, groups, X_test, transform="sqrt",
        ckpt_name="v5_3_pipe_b", n_folds=n_folds)
    log(f"  Pipeline B CV: {cv_b:.4f}")

    # Pipeline C: log1p + Huber
    log("  [3/3] Pipeline C: log1p + Huber...")
    lgb_huber = cfg.LGB_PARAMS.copy()
    lgb_huber["objective"] = "huber"
    lgb_huber["alpha"] = 1.35
    lgb_huber["metric"] = "mae"

    xgb_huber = cfg.XGB_PARAMS.copy()
    xgb_huber["objective"] = "reg:pseudohubererror"
    xgb_huber["huber_slope"] = 1.35

    cat_huber = cfg.CAT_PARAMS.copy()
    cat_huber["loss_function"] = "Huber:delta=1.35"
    cat_huber["eval_metric"] = "MAE"

    cv_c, test_c, oof_c = run_gbdt_cv(
        X, y_raw, groups, X_test, transform="log1p",
        lgb_params=lgb_huber, xgb_params=xgb_huber, cat_params=cat_huber,
        ckpt_name="v5_3_pipe_c", n_folds=n_folds)
    log(f"  Pipeline C CV: {cv_c:.4f}")

    # TabNet (v5-1 캐시)
    oof_tab = V5_CACHE.get("v1_oof_tab")
    test_tab = V5_CACHE.get("v1_test_tab")

    # Multi-way blend
    oof_list = [oof_a, oof_b, oof_c]
    test_list = [test_a, test_b, test_c]
    names = ["log1p+MAE", "sqrt+MAE", "log1p+Huber"]
    if oof_tab is not None:
        oof_list.append(oof_tab)
        test_list.append(test_tab)
        names.append("TabNet")

    w, cv_ens = optimize_blend_multi(oof_list, y_raw, names)
    test_ens = sum(w[i] * test_list[i] for i in range(len(w)))
    test_pp = apply_p99_clipping(test_ens)
    save_submission(test_fe, test_pp, "v5-3")

    V5_CACHE["v3_oof_sqrt"] = oof_b
    V5_CACHE["v3_test_sqrt"] = test_b
    V5_CACHE["v3_oof_huber"] = oof_c
    V5_CACHE["v3_test_huber"] = test_c

    log(f"v5-3 완료 -- A:{cv_a:.4f} B:{cv_b:.4f} C:{cv_c:.4f} | ENS:{cv_ens:.4f} ({(time.time()-t0)/60:.1f}분)")
    return cv_ens


# ══════════════════════════════════════════════════════════════════
# ── v5-4: High-Load Regime Features ──────────────────────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_4(data):
    log("=" * 60)
    log("v5-4: High-Load Regime 특화 피처")
    log("=" * 60)
    t0 = time.time()

    train_fe, test_fe = data["train_fe"], data["test_fe"]
    feature_cols = data["feature_cols"]
    sanity = data.get("sanity", False)
    n_folds = 2 if sanity else 5
    max_ep = 10 if sanity else 200
    seeds = [42] if sanity else [42, 123, 2026]

    # 1) 피처 추가
    log("  [1/4] 고부하 피처 추가...")
    train_hl, test_hl, ext_cols = add_highload_features(train_fe, test_fe, feature_cols)

    X = train_hl[ext_cols]
    y_raw = train_hl[cfg.TARGET].values
    groups = train_hl[cfg.GROUP_COL].values
    X_test = test_hl[ext_cols]

    # 2) Adversarial check
    log("  [2/4] Adversarial check...")
    new_cols = [c for c in ext_cols if c not in feature_cols]
    problematic, auc_diff = adversarial_check_features(train_hl, test_hl, feature_cols, new_cols)

    # 구분력이 크게 높아지는 피처 제거
    if problematic:
        log(f"  제거 피처: {problematic}")
        ext_cols = [c for c in ext_cols if c not in problematic]
        X = train_hl[ext_cols]
        X_test = test_hl[ext_cols]
        log(f"  최종 피처 수: {len(ext_cols)}")

    # 3) Multi-seed GBDT + adversarial weights (v5-2에서 가져오기)
    log("  [3/4] Multi-seed GBDT...")
    weights = V5_CACHE.get("adv_weights")
    cv_gbdt, test_gbdt, oof_gbdt = run_gbdt_multiseed(
        X, y_raw, groups, X_test, seeds=seeds,
        weights=weights, ckpt_prefix="v5_4_gbdt", n_folds=n_folds)

    # 4) Multi-seed TabNet (StandardScaler 적용 — 신규 피처 스케일 차이 보정)
    log("  [4/4] Multi-seed TabNet (scaled)...")
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_arr = scaler.fit_transform(X.fillna(0)).astype(np.float32)
    Xt_arr = scaler.transform(X_test.fillna(0)).astype(np.float32)
    cv_tab, test_tab, oof_tab = run_tabnet_multiseed(
        X_arr, y_raw, groups, Xt_arr, seeds=seeds,
        ckpt_prefix="v5_4_tabnet_sc", n_folds=n_folds, max_epochs=max_ep)

    best_a, cv_ens = optimize_blend_2way(oof_gbdt, oof_tab, y_raw)
    test_ens = best_a * test_gbdt + (1 - best_a) * test_tab
    test_pp = apply_p99_clipping(test_ens)
    save_submission(test_fe, test_pp, "v5-4")

    V5_CACHE["v4_oof_gbdt"] = oof_gbdt
    V5_CACHE["v4_test_gbdt"] = test_gbdt
    V5_CACHE["v4_oof_tab"] = oof_tab
    V5_CACHE["v4_test_tab"] = test_tab

    log(f"v5-4 완료 -- GBDT:{cv_gbdt:.4f} | Tab:{cv_tab:.4f} | ENS:{cv_ens:.4f} ({(time.time()-t0)/60:.1f}분)")
    return cv_ens


# ══════════════════════════════════════════════════════════════════
# ── v5-5: FT-Transformer ────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_5(data):
    log("=" * 60)
    log("v5-5: FT-Transformer 앙상블 다양성")
    log("=" * 60)
    t0 = time.time()

    train_fe, test_fe = data["train_fe"], data["test_fe"]
    feature_cols = data["feature_cols"]
    sanity = data.get("sanity", False)
    n_folds = 2 if sanity else 5
    max_ep = 10 if sanity else 100
    seeds = [42] if sanity else [42, 123, 2026]

    X = train_fe[feature_cols]
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values

    X_arr = X.fillna(0).values.astype(np.float32)
    Xt_arr = test_fe[feature_cols].fillna(0).values.astype(np.float32)

    # Multi-seed FT-Transformer
    log("  [1/2] Multi-seed FT-Transformer...")
    cv_ft, test_ft, oof_ft = run_ft_multiseed(
        X_arr, y_raw, groups, Xt_arr, seeds=seeds,
        ckpt_prefix="v5_5_ft", n_folds=n_folds, max_epochs=max_ep,
        batch_size=512)

    # 3-way blend: GBDT + TabNet + FT-Transformer
    log("  [2/2] 3-way blend...")
    oof_gbdt = V5_CACHE.get("v1_oof_gbdt")
    test_gbdt = V5_CACHE.get("v1_test_gbdt")
    oof_tab = V5_CACHE.get("v1_oof_tab")
    test_tab = V5_CACHE.get("v1_test_tab")

    if oof_gbdt is not None and oof_tab is not None:
        w, cv_ens = optimize_blend_multi(
            [oof_gbdt, oof_tab, oof_ft], y_raw,
            ["GBDT", "TabNet", "FT-Trans"])
        test_ens = w[0]*test_gbdt + w[1]*test_tab + w[2]*test_ft
    else:
        log("  v5-1 캐시 없음 -> FT + GBDT 2-way")
        cv_gbdt_s, test_gbdt_s, oof_gbdt_s = run_gbdt_cv(
            X, y_raw, groups, test_fe[feature_cols],
            ckpt_name="v5_5_gbdt_fallback", n_folds=n_folds)
        best_a, cv_ens = optimize_blend_2way(oof_gbdt_s, oof_ft, y_raw)
        test_ens = best_a * test_gbdt_s + (1 - best_a) * test_ft

    test_pp = apply_p99_clipping(test_ens)
    save_submission(test_fe, test_pp, "v5-5")

    V5_CACHE["v5_oof_ft"] = oof_ft
    V5_CACHE["v5_test_ft"] = test_ft

    log(f"v5-5 완료 -- FT:{cv_ft:.4f} | ENS:{cv_ens:.4f} ({(time.time()-t0)/60:.1f}분)")
    return cv_ens


# ══════════════════════════════════════════════════════════════════
# ── v5-6: Grand Ensemble + Distribution Calibration ──────────────
# ══════════════════════════════════════════════════════════════════
def run_v5_6(data):
    log("=" * 60)
    log("v5-6: Grand Ensemble + Distribution Calibration")
    log("=" * 60)
    t0 = time.time()

    train_fe = data["train_fe"]
    test_fe = data["test_fe"]
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values

    # 수집 가능한 모든 base model
    oof_models, test_models, model_names = [], [], []
    cache_map = [
        ("v1_oof_gbdt", "v1_test_gbdt", "v1-GBDT"),
        ("v1_oof_tab", "v1_test_tab", "v1-TabNet"),
        ("v2_oof_gbdt", "v2_test_gbdt", "v2-GBDT(adv)"),
        ("v3_oof_sqrt", "v3_test_sqrt", "v3-sqrt"),
        ("v3_oof_huber", "v3_test_huber", "v3-Huber"),
        ("v4_oof_gbdt", "v4_test_gbdt", "v4-GBDT(HL)"),
        ("v4_oof_tab", "v4_test_tab", "v4-TabNet(HL)"),
        ("v5_oof_ft", "v5_test_ft", "v5-FT-Trans"),
    ]
    for oof_key, test_key, name in cache_map:
        if oof_key in V5_CACHE and test_key in V5_CACHE:
            oof_models.append(V5_CACHE[oof_key])
            test_models.append(V5_CACHE[test_key])
            model_names.append(name)

    n_models = len(oof_models)
    log(f"  수집된 base models: {n_models}개 — {model_names}")

    if n_models < 2:
        log("  경고: base model 2개 미만 -> 건너뜀")
        return None

    # 1) Nelder-Mead blend
    log("  [1/3] Multi-way Nelder-Mead blend...")
    w_nm, cv_nm = optimize_blend_multi(oof_models, y_raw, model_names)

    # 2) ElasticNet stacking
    log("  [2/3] ElasticNet stacking...")
    meta_tr = np.column_stack(oof_models)
    meta_te = np.column_stack(test_models)

    scaler = StandardScaler()
    X_m = scaler.fit_transform(meta_tr)
    X_mt = scaler.transform(meta_te)

    gkf = GroupKFold(n_splits=5)
    oof_en = np.zeros(len(y_raw))
    test_en = np.zeros(len(meta_te))
    for _, (tr, va) in enumerate(gkf.split(X_m, groups=groups)):
        en = ElasticNetCV(
            l1_ratio=[0.1, 0.5, 0.9], alphas=[0.001, 0.01, 0.1, 1.0],
            cv=3, max_iter=5000, random_state=42)
        en.fit(X_m[tr], y_raw[tr])
        oof_en[va] = en.predict(X_m[va])
        test_en += en.predict(X_mt) / 5
    oof_en = np.clip(oof_en, 0, None)
    test_en = np.clip(test_en, 0, None)
    cv_en = mean_absolute_error(y_raw, oof_en)
    log(f"  ElasticNet CV: {cv_en:.4f}")

    # 최적 선택
    if cv_nm <= cv_en:
        log(f"  Nelder-Mead({cv_nm:.4f}) <= ElasticNet({cv_en:.4f}) -> Nelder-Mead 사용")
        test_final = sum(w_nm[i] * test_models[i] for i in range(n_models))
        oof_final = sum(w_nm[i] * oof_models[i] for i in range(n_models))
        cv_final = cv_nm
    else:
        log(f"  ElasticNet({cv_en:.4f}) < Nelder-Mead({cv_nm:.4f}) -> ElasticNet 사용")
        test_final = test_en
        oof_final = oof_en
        cv_final = cv_en

    # 3) Distribution calibration
    log("  [3/3] Distribution calibration...")
    residuals = y_raw - oof_final
    # Quantile별 잔차 패턴 분석
    q_bins = np.percentile(oof_final, [0, 25, 50, 75, 100])
    corrections = []
    for i in range(4):
        mask = (oof_final >= q_bins[i]) & (oof_final < q_bins[i+1] + (1 if i == 3 else 0))
        if mask.sum() > 100:
            med_resid = np.median(residuals[mask])
            corrections.append((q_bins[i], q_bins[i+1], med_resid, mask.sum()))
            log(f"    Q{i+1} [{q_bins[i]:.1f}-{q_bins[i+1]:.1f}]: "
                f"median resid={med_resid:+.3f}  n={mask.sum()}")

    # 보정 적용 (보수적: 잔차의 50%만 보정)
    test_calibrated = test_final.copy()
    for lo, hi, resid, _ in corrections:
        if abs(resid) > 0.1:  # 의미있는 보정만
            mask_test = (test_final >= lo) & (test_final < hi + (1 if hi == q_bins[-1] else 0))
            test_calibrated[mask_test] += resid * 0.5
            log(f"    보정: [{lo:.1f}-{hi:.1f}] += {resid*0.5:+.3f}")
    test_calibrated = np.clip(test_calibrated, 0, None)

    # 보정 효과 비교 (OOF 기준)
    oof_cal = oof_final.copy()
    for lo, hi, resid, _ in corrections:
        if abs(resid) > 0.1:
            mask_oof = (oof_final >= lo) & (oof_final < hi + (1 if hi == q_bins[-1] else 0))
            oof_cal[mask_oof] += resid * 0.5
    oof_cal = np.clip(oof_cal, 0, None)
    cv_cal = mean_absolute_error(y_raw, oof_cal)
    log(f"  Calibration 효과: {cv_final:.4f} -> {cv_cal:.4f}")

    if cv_cal < cv_final:
        log("  -> Calibration 적용")
        test_out = test_calibrated
        cv_out = cv_cal
    else:
        log("  -> Calibration 효과 없음, 원본 사용")
        test_out = test_final
        cv_out = cv_final

    test_pp = apply_p99_clipping(test_out)
    save_submission(test_fe, test_pp, "v5-6")

    log(f"v5-6 완료 -- NM:{cv_nm:.4f} | EN:{cv_en:.4f} | 최종:{cv_out:.4f} ({(time.time()-t0)/60:.1f}분)")
    return cv_out


# ══════════════════════════════════════════════════════════════════
# ── 메인 ─────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════
def main():
    parser = argparse.ArgumentParser(description="v5 실험 (Pseudo-labeling 완전 제거)")
    parser.add_argument("--sanity", action="store_true", help="Sanity check (2-fold, 축소)")
    parser.add_argument("--exp", type=str, default=None,
                        help="특정 실험만 실행 (예: 1, 1-3, 3,5,6)")
    args = parser.parse_args()

    # sanity 모드면 체크포인트 디렉토리 분리
    global CKPT_DIR
    if args.sanity:
        CKPT_DIR = os.path.join(os.path.dirname(__file__), "checkpoints_v5_sanity")
        os.makedirs(CKPT_DIR, exist_ok=True)

    log("=" * 60)
    log("v5 실험 시작 — Pseudo-labeling 완전 제거")
    log(f"EXP-D 기준 (유효 baseline): LB MAE = {EXPD_LB}")
    if args.sanity:
        log("!! SANITY CHECK 모드 (2-fold, 축소 학습)")
    log("=" * 60)

    # 실행할 실험 결정
    all_exps = [
        ("v5-1", run_v5_1),
        ("v5-2", run_v5_2),
        ("v5-3", run_v5_3),
        ("v5-4", run_v5_4),
        ("v5-5", run_v5_5),
        ("v5-6", run_v5_6),
    ]

    if args.exp:
        # "1-3" → [1,2,3], "3,5,6" → [3,5,6]
        selected = set()
        for part in args.exp.split(","):
            if "-" in part:
                a, b = part.split("-")
                selected.update(range(int(a), int(b) + 1))
            else:
                selected.add(int(part))
        experiments = [(n, f) for i, (n, f) in enumerate(all_exps) if (i + 1) in selected]
        log(f"선택 실험: {[n for n, _ in experiments]}")
    else:
        experiments = all_exps

    # 데이터 로딩
    log("데이터 로딩...")
    train_raw, test_raw, layout = load_raw_data()
    train_fe, test_fe, feature_cols = build_features(train_raw, test_raw, layout)
    log(f"피처 수: {len(feature_cols)}")

    data = {
        "train_fe": train_fe,
        "test_fe": test_fe,
        "feature_cols": feature_cols,
        "sanity": args.sanity,
    }

    # 실험 실행
    for name, func in experiments:
        try:
            cv_mae = func(data)
            RESULTS[name] = cv_mae
        except Exception as e:
            log(f"\n{'!'*60}")
            log(f"{name} 실패: {e}")
            traceback.print_exc()
            log("!" * 60)
            RESULTS[name] = None

    # 비교표
    print(f"\n{'='*60}")
    print("  v5 실험 결과 비교")
    print(f"{'='*60}")
    print(f"  {'실험':<16} {'CV MAE':>10} {'vs EXP-D LB':>12}")
    print(f"  {'-'*40}")
    print(f"  {'EXP-D (기준)':<16} {'8.8827':>10} {'LB=10.2428':>12}")
    for name, mae in RESULTS.items():
        if mae is not None:
            print(f"  {name:<16} {mae:>10.4f}")
        else:
            print(f"  {name:<16} {'FAILED':>10}")
    print(f"{'='*60}")
    print(f"\n제출 파일: submissions/v5-{{1..6}}_submission.csv")


if __name__ == "__main__":
    main()


---
## run_experiments_strict.py (strict 피처)


In [ ]:
# run_experiments_strict.py
# ============================================================
# 아래 코드를 run_experiments_strict.py 파일로 저장하세요

"""
File: run_experiments_strict.py
"""


In [ ]:
"""
Strict-clean experiments for code-verifiable submissions.

Rules enforced:
- test.csv is never used for model fitting, CV tuning, feature selection,
  sample weighting, scaler/imputer fitting, or clipping-threshold selection.
- test.csv is loaded only after train-side features and thresholds are fixed,
  then used for final inference.
- no pseudo-labeling, no train/test adversarial modeling, no Public-LB tuning.
"""

import argparse
import os
import time

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

import config as cfg
from dataset import (
    add_expanding_features,
    add_interaction_features,
    add_lag_features,
    add_rolling_features,
    add_scenario_stats,
    add_timestep,
    merge_layout,
)
import run_experiments_v5 as v5


BASE_DIR = os.path.dirname(os.path.abspath(__file__))
STRICT_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_strict")
STRICT_SANITY_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_strict_sanity")


def log(msg):
    v5.log(msg)


def load_train_only():
    train = pd.read_csv(cfg.TRAIN_FILE)
    layout = pd.read_csv(cfg.LAYOUT_FILE)
    return train, layout


def load_test_only():
    test = pd.read_csv(cfg.TEST_FILE)
    layout = pd.read_csv(cfg.LAYOUT_FILE)
    return test, layout


def build_features_one(df, layout, has_target):
    """Build features for one split only. No train/test concat."""
    out = df.copy()
    out = merge_layout(out, layout)
    out = add_timestep(out)
    out = add_lag_features(out)
    out = add_rolling_features(out)
    out = add_expanding_features(out)
    out = add_interaction_features(out)
    out = add_scenario_stats(out)

    exclude = set(cfg.ID_COLS + ["_is_train"])
    if has_target:
        exclude.add(cfg.TARGET)
    feature_cols = [c for c in out.columns if c not in exclude]
    return out, feature_cols


def align_test_features(test_fe, feature_cols):
    missing = [c for c in feature_cols if c not in test_fe.columns]
    if missing:
        raise ValueError(f"test missing features: {missing[:10]}")
    return test_fe[feature_cols]


def take_sanity_subset(train_fe, max_groups=1200):
    groups = train_fe[cfg.GROUP_COL].drop_duplicates().iloc[:max_groups]
    return train_fe[train_fe[cfg.GROUP_COL].isin(groups)].reset_index(drop=True)


def add_highload_strict(train_fe, test_fe, feature_cols):
    """v5-4 style high-load features without adversarial check/selection."""
    train_out = train_fe.copy()
    test_out = test_fe.copy()
    thresholds = {
        "order_p75": np.nanpercentile(train_out["order_inflow_15m"], 75),
        "battery_p90": np.nanpercentile(train_out["low_battery_ratio"], 90),
        "congestion_p90": np.nanpercentile(train_out["congestion_score"], 90),
    }
    log(
        "  strict thresholds: "
        f"order_P75={thresholds['order_p75']:.2f} "
        f"battery_P90={thresholds['battery_p90']:.4f} "
        f"congestion_P90={thresholds['congestion_p90']:.2f}"
    )

    for df in (train_out, test_out):
        rt = df["robot_total"].replace(0, np.nan)
        cc = df["charger_count"].replace(0, np.nan)

        df["order_surge"] = (df["order_inflow_15m"] > thresholds["order_p75"]).astype(np.float32)
        df["battery_critical"] = (df["low_battery_ratio"] > thresholds["battery_p90"]).astype(np.float32)
        df["congestion_critical"] = (df["congestion_score"] > thresholds["congestion_p90"]).astype(np.float32)
        df["robot_capacity_used"] = (df["robot_active"] + df["robot_charging"]) / rt
        df["stress_index"] = df["order_inflow_15m"] * df["congestion_score"] * df["low_battery_ratio"]
        df["bottleneck_score"] = df["max_zone_density"] * (1 - df["robot_idle"] / rt)
        df["recovery_pressure"] = df["charge_queue_length"] * df["avg_charge_wait"] / (cc + 1)
        df["demand_supply_gap"] = df["order_inflow_15m"] - df["robot_active"]
        df["cascade_risk"] = df["fault_count_15m"] * df["congestion_score"] * df["blocked_path_15m"]
        df["pack_station_per_robot"] = df["pack_station_count"] / rt
        df["charger_per_robot"] = df["charger_count"] / rt

        grp = df.groupby(cfg.GROUP_COL)["stress_index"]
        df["stress_acceleration"] = df["stress_index"] - grp.shift(1)
        shifted = grp.shift(1)
        df["sustained_stress"] = shifted.groupby(df[cfg.GROUP_COL]).rolling(
            3, min_periods=1
        ).mean().reset_index(level=0, drop=True)
        sc_max = grp.transform("max").replace(0, np.nan)
        df["peak_stress_ratio"] = df["stress_index"] / sc_max

    new_cols = [
        "order_surge", "battery_critical", "congestion_critical", "robot_capacity_used",
        "stress_index", "bottleneck_score", "recovery_pressure", "demand_supply_gap",
        "cascade_risk", "pack_station_per_robot", "charger_per_robot",
        "stress_acceleration", "sustained_stress", "peak_stress_ratio",
    ]
    return train_out, test_out, feature_cols + new_cols


def add_pressure_strict(train_fe, test_fe, feature_cols, mode):
    """v6 pressure features, but built without any test-driven selection/weights."""
    train_out, test_out, cols = add_highload_strict(train_fe, test_fe, feature_cols)
    new_cols = []

    for df in (train_out, test_out):
        robot_total = df["robot_total"].replace(0, np.nan)
        robot_active = df["robot_active"].replace(0, np.nan)
        charger_count = df["charger_count"].replace(0, np.nan)
        pack_station = df["pack_station_count"].replace(0, np.nan)

        df["order_per_robot"] = df["order_inflow_15m"] / robot_total
        df["order_per_active_robot"] = df["order_inflow_15m"] / robot_active
        df["urgent_order_load"] = df["order_inflow_15m"] * df["urgent_order_ratio"]
        df["heavy_order_load"] = df["order_inflow_15m"] * df["heavy_item_ratio"]
        df["sku_complexity_load"] = df["unique_sku_15m"] * (1 + df["sku_concentration"])
        df["capacity_gap"] = df["order_inflow_15m"] - (df["robot_active"] + df["robot_idle"])
        df["idle_capacity_buffer"] = df["robot_idle"] / robot_total
        df["charger_queue_per_charger"] = df["charge_queue_length"] / charger_count
        df["charge_pressure"] = df["charge_queue_length"] * df["avg_charge_wait"] / (charger_count + 1)
        df["battery_deficit_pressure"] = (100 - df["battery_mean"]) * df["low_battery_ratio"]
        df["charging_capacity_pressure"] = df["robot_charging"] / charger_count
        df["pack_pressure"] = df["order_inflow_15m"] * df["pack_utilization"] / (pack_station + 1)
        df["dock_pressure"] = df["loading_dock_util"] * df["outbound_truck_wait_min"]
        df["traffic_pressure"] = df["congestion_score"] * (
            df["blocked_path_15m"] + df["near_collision_15m"] + df["intersection_wait_time_avg"]
        )
        df["conveyor_pack_pressure"] = df["pack_utilization"] / (df["conveyor_speed_mps"] + 0.1)
        df["staging_pressure"] = df["staging_area_util"] * df["order_inflow_15m"]

        if mode == "expanded":
            df["staffing_pressure"] = df["order_inflow_15m"] / (df["staff_on_floor"] + 1)
            df["forecast_miss_load"] = df["order_inflow_15m"] * (1 - df["daily_forecast_accuracy"])
            df["agv_failure_load"] = df["order_inflow_15m"] * (1 - df["agv_task_success_rate"])
            df["system_drag"] = (
                df["wms_response_time_ms"]
                * (df["network_latency_ms"] + 1)
                * (1 + df["scanner_error_rate"])
            )
            df["inventory_congestion"] = (
                df["storage_density_pct"] * df["vertical_utilization"] * df["aisle_traffic_score"]
            )
            df["robot_health_drag"] = (
                (df["fleet_age_months_avg"] / 60.0)
                * (100 - df["maintenance_schedule_score"])
                * (100 - df["robot_calibration_score"])
                / 100.0
            )
            df["pick_wave_pressure"] = (
                df["order_wave_count"] * df["pick_list_length_avg"] * (1 + df["bulk_order_ratio"])
            )
            df["express_pressure"] = (
                df["express_lane_util"] * df["urgent_order_ratio"] * df["order_inflow_15m"]
            )
            df["return_rework_pressure"] = (
                df["return_order_ratio"] * df["quality_check_rate"] * df["order_inflow_15m"]
            )
            df["cold_chain_load"] = df["cold_chain_ratio"] * df["order_inflow_15m"]

        for col in ["capacity_gap", "charge_pressure", "pack_pressure", "traffic_pressure"]:
            grp = df.groupby(cfg.GROUP_COL)[col]
            df[f"{col}_diff1"] = df[col] - grp.shift(1)
            shifted = grp.shift(1)
            df[f"{col}_rmean3"] = shifted.groupby(df[cfg.GROUP_COL]).rolling(
                3, min_periods=1
            ).mean().reset_index(level=0, drop=True)

    focused_cols = [
        "order_per_robot", "order_per_active_robot", "urgent_order_load", "heavy_order_load",
        "sku_complexity_load", "capacity_gap", "idle_capacity_buffer",
        "charger_queue_per_charger", "charge_pressure", "battery_deficit_pressure",
        "charging_capacity_pressure", "pack_pressure", "dock_pressure", "traffic_pressure",
        "conveyor_pack_pressure", "staging_pressure",
    ]
    expanded_cols = [
        "staffing_pressure", "forecast_miss_load", "agv_failure_load", "system_drag",
        "inventory_congestion", "robot_health_drag", "pick_wave_pressure", "express_pressure",
        "return_rework_pressure", "cold_chain_load",
    ]
    temporal_cols = []
    for c in ["capacity_gap", "charge_pressure", "pack_pressure", "traffic_pressure"]:
        temporal_cols.extend([f"{c}_diff1", f"{c}_rmean3"])

    new_cols = focused_cols + temporal_cols
    if mode == "expanded":
        new_cols += expanded_cols

    out_cols = list(cols)
    seen = set(out_cols)
    for col in new_cols:
        if col not in seen:
            out_cols.append(col)
            seen.add(col)
    return train_out, test_out, out_cols


def oof_clip(oof_pred, test_pred, factor=1.10):
    clip_val = np.percentile(oof_pred, 99) * factor
    oof_out = np.clip(oof_pred, 0, clip_val)
    test_out = np.clip(test_pred, 0, clip_val)
    return oof_out, test_out, clip_val


def save_submission(test_fe, preds, name):
    sub = pd.DataFrame({cfg.ID_COL: test_fe[cfg.ID_COL], cfg.TARGET: np.clip(preds, 0, None)})
    sample = pd.read_csv(cfg.SAMPLE_SUB_FILE)
    if len(sub) != len(sample):
        raise ValueError(f"submission rows mismatch: {len(sub)} != {len(sample)}")
    path = os.path.join(cfg.SUBMISSION_DIR, f"{name}_submission.csv")
    sub.to_csv(path, index=False)
    log(f"  저장: {path} mean={sub[cfg.TARGET].mean():.2f} std={sub[cfg.TARGET].std():.2f} max={sub[cfg.TARGET].max():.2f}")
    return path


def gbdt_params(sanity=False):
    lgb_p = cfg.LGB_PARAMS.copy()
    xgb_p = cfg.XGB_PARAMS.copy()
    cat_p = cfg.CAT_PARAMS.copy()
    if sanity:
        lgb_p["n_estimators"] = cfg.SANITY_N_ESTIMATORS
        xgb_p["n_estimators"] = cfg.SANITY_N_ESTIMATORS
        cat_p["iterations"] = cfg.SANITY_N_ESTIMATORS
    return lgb_p, xgb_p, cat_p


def build_strict_data(sanity=False):
    log("train 데이터 로딩 및 train-only feature build...")
    train_raw, layout = load_train_only()
    train_fe, feature_cols = build_features_one(train_raw, layout, has_target=True)
    if sanity:
        train_fe = take_sanity_subset(train_fe)
        log(f"  sanity train subset: {train_fe.shape}")

    log("test 데이터 로딩 및 fixed transform 적용...")
    test_raw, layout_test = load_test_only()
    test_fe, test_cols = build_features_one(test_raw, layout_test, has_target=False)
    missing = [c for c in feature_cols if c not in test_cols]
    if missing:
        raise ValueError(f"feature mismatch: {missing[:10]}")

    log(f"  strict base features: {len(feature_cols)} train={train_fe.shape} test={test_fe.shape}")
    return train_fe, test_fe, feature_cols


def run_experiment(name, train_fe, test_fe, feature_cols, mode, sanity=False, skip_tabnet=False):
    log("=" * 60)
    log(f"{name}: strict-clean mode={mode}")
    log("=" * 60)

    if mode == "base":
        train_x, test_x, cols = train_fe.copy(), test_fe.copy(), list(feature_cols)
    elif mode in ("focused", "expanded"):
        train_x, test_x, cols = add_pressure_strict(train_fe, test_fe, feature_cols, mode=mode)
    else:
        raise ValueError(mode)

    X = train_x[cols]
    X_test = align_test_features(test_x, cols)
    y_raw = train_x[cfg.TARGET].values
    groups = train_x[cfg.GROUP_COL].values

    seeds = [42] if sanity else [42, 123, 2026]
    n_folds = 2 if sanity else cfg.N_FOLDS
    early_stop = 30 if sanity else cfg.EARLY_STOPPING_ROUNDS
    max_epochs = 10 if sanity else 200
    lgb_p, xgb_p, cat_p = gbdt_params(sanity=sanity)

    t0 = time.time()
    log("  [1/3] strict GBDT 학습...")
    cv_gbdt, test_gbdt, oof_gbdt = v5.run_gbdt_multiseed(
        X, y_raw, groups, X_test,
        seeds=seeds,
        lgb_params=lgb_p,
        xgb_params=xgb_p,
        cat_params=cat_p,
        n_folds=n_folds,
        early_stop=early_stop,
        ckpt_prefix=f"{name}_gbdt",
    )

    models = [("gbdt", cv_gbdt, oof_gbdt, test_gbdt)]
    if not skip_tabnet:
        log("  [2/3] strict TabNet 학습 (scaler fit=train only)...")
        scaler = StandardScaler()
        X_arr = scaler.fit_transform(X.fillna(0)).astype(np.float32)
        Xt_arr = scaler.transform(X_test.fillna(0)).astype(np.float32)
        cv_tab, test_tab, oof_tab = v5.run_tabnet_multiseed(
            X_arr, y_raw, groups, Xt_arr,
            seeds=seeds,
            ckpt_prefix=f"{name}_tabnet",
            n_folds=n_folds,
            max_epochs=max_epochs,
        )
        models.append(("tabnet", cv_tab, oof_tab, test_tab))

    log("  [3/3] train OOF 기준 blend/clipping...")
    if len(models) == 1:
        model_name, cv, oof, test_pred = models[0]
        alpha = 1.0
    else:
        _, _, oof_a, test_a = models[0]
        _, _, oof_b, test_b = models[1]
        alpha, cv = v5.optimize_blend_2way(oof_a, oof_b, y_raw)
        oof = alpha * oof_a + (1 - alpha) * oof_b
        test_pred = alpha * test_a + (1 - alpha) * test_b

    oof_pp, test_pp, clip_val = oof_clip(oof, test_pred, factor=1.10)
    cv_pp = mean_absolute_error(y_raw, oof_pp)
    path = save_submission(test_fe, test_pp, name.replace("_", "-"))
    log(
        f"{name} 완료 -- CV raw={cv:.4f} CV clipped={cv_pp:.4f} "
        f"alpha={alpha:.3f} clip_from_oof={clip_val:.2f} path={path} "
        f"elapsed={(time.time() - t0) / 60:.1f}m"
    )
    return {"name": name, "cv": cv, "cv_clipped": cv_pp, "path": path}


def parse_exp(arg):
    all_exps = [
        ("strict_1_base", "base"),
        ("strict_2_focused", "focused"),
        ("strict_3_expanded", "expanded"),
    ]
    if not arg:
        return all_exps
    selected = set()
    for part in arg.split(","):
        if "-" in part:
            a, b = part.split("-")
            selected.update(range(int(a), int(b) + 1))
        else:
            selected.add(int(part))
    return [item for i, item in enumerate(all_exps, start=1) if i in selected]


def main():
    parser = argparse.ArgumentParser(description="strict-clean smart warehouse experiments")
    parser.add_argument("--sanity", action="store_true")
    parser.add_argument("--exp", type=str, default=None)
    parser.add_argument("--skip-tabnet", action="store_true")
    args = parser.parse_args()

    v5.CKPT_DIR = STRICT_SANITY_CKPT_DIR if args.sanity else STRICT_CKPT_DIR
    os.makedirs(v5.CKPT_DIR, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)

    log("=" * 60)
    log("strict-clean 실험 시작")
    if args.sanity:
        log("!! SANITY CHECK 모드")
    log(f"체크포인트: {v5.CKPT_DIR}")
    log("=" * 60)

    train_fe, test_fe, feature_cols = build_strict_data(sanity=args.sanity)
    results = []
    for name, mode in parse_exp(args.exp):
        results.append(run_experiment(
            name, train_fe, test_fe, feature_cols,
            mode=mode,
            sanity=args.sanity,
            skip_tabnet=args.skip_tabnet,
        ))

    print("\n" + "=" * 60)
    print("strict-clean 결과 요약")
    print("=" * 60)
    for r in results:
        print(f"{r['name']:<20} CV={r['cv']:.4f} clipped={r['cv_clipped']:.4f} {r['path']}")
    print("=" * 60)


if __name__ == "__main__":
    main()


---
## run_experiments_attack.py (attack 피처)


In [ ]:
# run_experiments_attack.py
# ============================================================
# 아래 코드를 run_experiments_attack.py 파일로 저장하세요

"""
File: run_experiments_attack.py
"""


In [ ]:
"""
Strict-clean attack-track experiments.

This reimplements the v7 lead/forward idea without train/test concat and
without v5/v6/v7 cached submissions. Test rows are used only for final
feature transformation and inference.

Risk note:
- Lead/forward covariates use later timesteps from the same scenario.
- They do not use test targets or pseudo-labels, but they assume all scenario
  covariates are available at inference time.
"""

import argparse
import os
import pickle
import time

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

import config as cfg
import run_experiments_strict as strict
import run_experiments_v5 as v5


BASE_DIR = os.path.dirname(os.path.abspath(__file__))
ATTACK_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_attack")
ATTACK_SANITY_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_attack_sanity")
STRICT_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_strict")


CORE_LEAD_FEATURES = [
    "order_inflow_15m", "congestion_score", "low_battery_ratio", "battery_mean",
    "robot_idle", "robot_charging", "robot_active", "max_zone_density",
    "charge_queue_length", "avg_charge_wait", "pack_utilization",
    "loading_dock_util", "blocked_path_15m", "near_collision_15m",
]

OPS_LEAD_FEATURES = [
    "urgent_order_ratio", "heavy_item_ratio", "unique_sku_15m", "sku_concentration",
    "fault_count_15m", "avg_recovery_time", "staging_area_util",
    "intersection_wait_time_avg", "aisle_traffic_score", "outbound_truck_wait_min",
    "wms_response_time_ms", "network_latency_ms",
]


def log(msg):
    v5.log(msg)


def append_unique(base_cols, new_cols):
    seen = set(base_cols)
    out = list(base_cols)
    for col in new_cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def add_lead_features_split(train_fe, test_fe, feature_cols, lead_mode="core", base_mode="focused"):
    """Add same-scenario forward covariates per split. No train/test concat."""
    if base_mode in ("focused", "expanded"):
        train_out, test_out, cols = strict.add_pressure_strict(
            train_fe, test_fe, feature_cols, mode=base_mode
        )
    elif base_mode == "base":
        train_out, test_out, cols = train_fe.copy(), test_fe.copy(), list(feature_cols)
    else:
        raise ValueError(base_mode)

    lead_features = list(CORE_LEAD_FEATURES)
    if lead_mode == "ops":
        lead_features += OPS_LEAD_FEATURES
    elif lead_mode != "core":
        raise ValueError(lead_mode)

    new_cols = []
    for df in (train_out, test_out):
        grp = df.groupby(cfg.GROUP_COL, sort=False)
        for col in lead_features:
            if col not in df.columns:
                continue
            g = grp[col]
            lead1 = f"{col}_lead1"
            lead2 = f"{col}_lead2"
            delta = f"{col}_lead1_delta"
            fmean2 = f"{col}_fmean2"
            df[lead1] = g.shift(-1)
            df[lead2] = g.shift(-2)
            df[delta] = df[lead1] - df[col]
            df[fmean2] = (df[col] + df[lead1]) / 2.0
            new_cols.extend([lead1, lead2, delta, fmean2])

        if {"order_inflow_15m_lead1", "congestion_score_lead1", "low_battery_ratio_lead1"}.issubset(df.columns):
            df["future_stress_30m"] = (
                df["order_inflow_15m_lead1"].fillna(df["order_inflow_15m"])
                * df["congestion_score_lead1"].fillna(df["congestion_score"])
                * df["low_battery_ratio_lead1"].fillna(df["low_battery_ratio"])
            )
            if "stress_index" in df.columns:
                df["future_stress_delta"] = df["future_stress_30m"] - df["stress_index"]
                new_cols.append("future_stress_delta")
            new_cols.append("future_stress_30m")

        if {"charge_queue_length_lead1", "avg_charge_wait_lead1", "charger_count"}.issubset(df.columns):
            df["future_charge_pressure"] = (
                df["charge_queue_length_lead1"].fillna(df["charge_queue_length"])
                * df["avg_charge_wait_lead1"].fillna(df["avg_charge_wait"])
                / (df["charger_count"].replace(0, np.nan) + 1)
            )
            new_cols.append("future_charge_pressure")

    final_cols = append_unique(cols, new_cols)
    log(
        f"  attack lead features base={base_mode} lead={lead_mode}: "
        f"{len(feature_cols)} -> {len(final_cols)} (+{len(final_cols) - len(feature_cols)})"
    )
    return train_out, test_out, final_cols


def load_strict_seed_bundle(prefix, seeds=(42, 123, 2026)):
    oofs, tests, cvs = [], [], []
    for seed in seeds:
        path = os.path.join(STRICT_CKPT_DIR, f"{prefix}_seed{seed}.pkl")
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        with open(path, "rb") as f:
            cv, test_pred, oof_pred = pickle.load(f)
        cvs.append(cv)
        tests.append(test_pred)
        oofs.append(oof_pred)
    oof_avg = np.mean(oofs, axis=0)
    test_avg = np.mean(tests, axis=0)
    return cvs, test_avg, oof_avg


def save_submission(test_fe, pred, name):
    sub = pd.DataFrame({
        cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
        cfg.TARGET: np.clip(pred, 0, None),
    })
    sample = pd.read_csv(cfg.SAMPLE_SUB_FILE)
    if list(sub.columns) != list(sample.columns):
        raise ValueError(f"column mismatch: {list(sub.columns)} != {list(sample.columns)}")
    if len(sub) != len(sample):
        raise ValueError(f"row mismatch: {len(sub)} != {len(sample)}")
    if not sub[cfg.ID_COL].astype(str).equals(sample[cfg.ID_COL].astype(str)):
        raise ValueError("ID order mismatch")
    if sub[cfg.TARGET].isna().any() or (sub[cfg.TARGET] < 0).any():
        raise ValueError("bad predictions")
    path = os.path.join(cfg.SUBMISSION_DIR, f"{name}_submission.csv")
    sub.to_csv(path, index=False, float_format="%.10f", lineterminator="\n")
    log(
        f"  저장: {path} mean={sub[cfg.TARGET].mean():.3f} "
        f"std={sub[cfg.TARGET].std():.3f} max={sub[cfg.TARGET].max():.3f}"
    )
    return path


def save_variants(name, test_fe, y_raw, test_gbdt, oof_gbdt, test_tab, oof_tab):
    out = []

    gbdt_cv = mean_absolute_error(y_raw, oof_gbdt)
    clip_g = np.percentile(oof_gbdt, 99) * 1.30
    gbdt_clip_cv = mean_absolute_error(y_raw, np.clip(oof_gbdt, 0, clip_g))
    out.append({
        "name": f"{name}_gbdt_clip130",
        "cv": gbdt_clip_cv,
        "path": save_submission(test_fe, np.clip(test_gbdt, 0, clip_g), f"{name}-gbdt-clip130"),
    })
    log(f"  {name} GBDT raw CV={gbdt_cv:.4f} clip130 CV={gbdt_clip_cv:.4f} clip={clip_g:.2f}")

    if len(oof_tab) != len(y_raw):
        log(
            f"  TabNet OOF length mismatch ({len(oof_tab)} != {len(y_raw)}); "
            "blend variants skipped. This is expected in sanity mode."
        )
        return out

    alpha, blend_cv = v5.optimize_blend_2way(oof_gbdt, oof_tab, y_raw)
    oof_blend = alpha * oof_gbdt + (1 - alpha) * oof_tab
    test_blend = alpha * test_gbdt + (1 - alpha) * test_tab

    for factor in [1.10, 1.30, None]:
        if factor is None:
            label = "noclip"
            pred = np.clip(test_blend, 0, None)
            cv = mean_absolute_error(y_raw, np.clip(oof_blend, 0, None))
        else:
            label = f"clip{int(factor * 100):03d}"
            clip_val = np.percentile(oof_blend, 99) * factor
            pred = np.clip(test_blend, 0, clip_val)
            cv = mean_absolute_error(y_raw, np.clip(oof_blend, 0, clip_val))
        out.append({
            "name": f"{name}_blend_{label}",
            "cv": cv,
            "path": save_submission(test_fe, pred, f"{name}-blend-{label}"),
            "alpha": alpha,
        })
        log(f"  {name} blend {label}: alpha={alpha:.3f} rawCV={blend_cv:.4f} cv={cv:.4f}")

    return out


def run_experiment(name, train_fe, test_fe, feature_cols, lead_mode, base_mode, sanity=False):
    log("=" * 60)
    log(f"{name}: attack base={base_mode} lead={lead_mode}")
    log("=" * 60)

    train_x, test_x, cols = add_lead_features_split(
        train_fe, test_fe, feature_cols, lead_mode=lead_mode, base_mode=base_mode
    )
    X = train_x[cols]
    X_test = strict.align_test_features(test_x, cols)
    y_raw = train_x[cfg.TARGET].values
    groups = train_x[cfg.GROUP_COL].values

    seeds = [42] if sanity else [42, 123, 2026]
    n_folds = 2 if sanity else cfg.N_FOLDS
    early_stop = 30 if sanity else cfg.EARLY_STOPPING_ROUNDS
    lgb_p, xgb_p, cat_p = strict.gbdt_params(sanity=sanity)

    log("  [1/3] attack GBDT 학습...")
    cv_gbdt, test_gbdt, oof_gbdt = v5.run_gbdt_multiseed(
        X, y_raw, groups, X_test,
        seeds=seeds,
        lgb_params=lgb_p,
        xgb_params=xgb_p,
        cat_params=cat_p,
        n_folds=n_folds,
        early_stop=early_stop,
        ckpt_prefix=f"{name}_gbdt",
    )

    log("  [2/3] strict2 TabNet clean 캐시 로드...")
    cvs_tab, test_tab, oof_tab = load_strict_seed_bundle(
        "strict_2_focused_tabnet", seeds=(42, 123, 2026)
    )
    log(f"  strict2 TabNet seed CV={['%.4f' % c for c in cvs_tab]}")

    log("  [3/3] OOF 기준 후보 저장...")
    variants = save_variants(name, test_fe, y_raw, test_gbdt, oof_gbdt, test_tab, oof_tab)
    return {"name": name, "cv_gbdt": cv_gbdt, "variants": variants}


def parse_exp(arg):
    exps = [
        ("attack_1_core_lead", "core", "focused"),
        ("attack_2_ops_lead", "ops", "focused"),
        ("attack_3_expanded_core_lead", "core", "expanded"),
        ("attack_4_expanded_ops_lead", "ops", "expanded"),
    ]
    if not arg:
        return exps[:2]
    selected = set()
    for part in arg.split(","):
        if "-" in part:
            a, b = part.split("-")
            selected.update(range(int(a), int(b) + 1))
        else:
            selected.add(int(part))
    return [item for i, item in enumerate(exps, start=1) if i in selected]


def main():
    parser = argparse.ArgumentParser(description="strict-clean attack-track experiments")
    parser.add_argument("--sanity", action="store_true")
    parser.add_argument("--exp", type=str, default=None)
    args = parser.parse_args()

    v5.CKPT_DIR = ATTACK_SANITY_CKPT_DIR if args.sanity else ATTACK_CKPT_DIR
    os.makedirs(v5.CKPT_DIR, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)

    log("=" * 60)
    log("strict-clean attack-track 시작")
    if args.sanity:
        log("!! SANITY CHECK 모드")
    log(f"체크포인트: {v5.CKPT_DIR}")
    log("=" * 60)

    train_fe, test_fe, feature_cols = strict.build_strict_data(sanity=args.sanity)
    results = []
    t0 = time.time()
    for name, lead_mode, base_mode in parse_exp(args.exp):
        results.append(run_experiment(
            name, train_fe, test_fe, feature_cols,
            lead_mode=lead_mode,
            base_mode=base_mode,
            sanity=args.sanity,
        ))

    print("\n" + "=" * 60)
    print("attack-track 결과 요약")
    print("=" * 60)
    for res in results:
        print(f"{res['name']:<32} GBDT CV={res['cv_gbdt']:.4f}")
        for var in res["variants"]:
            extra = f" alpha={var['alpha']:.3f}" if "alpha" in var else ""
            print(f"  {var['name']:<36} CV={var['cv']:.4f}{extra} {var['path']}")
    print(f"총 소요: {(time.time() - t0) / 60:.1f}분")
    print("=" * 60)


if __name__ == "__main__":
    main()


---
## run_experiments_v8.py (GBDT 798피처 학습)


In [ ]:
# run_experiments_v8.py
# ============================================================
# 아래 코드를 run_experiments_v8.py 파일로 저장하세요

"""
File: run_experiments_v8.py
"""


In [ ]:
"""
v8 — PB-style features + objective diversity + sample weighting.

Combines three improvements over the attack pipeline:
1. Rich PB-notebook-style feature engineering (~480 features)
2. Per-model objective/transform diversity (MAE raw, Huber log1p, MAE log1p)
3. Target-skew sample weighting (q90/q95/q99 bonus)

Strict-clean: test.csv used only for final inference.
"""

import argparse
import os
import pickle
import time

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

import config as cfg
import run_experiments_v5 as v5
import run_experiments_strict as strict
import run_experiments_attack as attack

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
V8_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8")
V8_SANITY_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8_sanity")


def log(msg):
    v5.log(msg)


# ── PB-style features ──────────────────────────────────────────────

def safe_divide(a, b):
    num = pd.Series(a, dtype="float64")
    den = pd.Series(b, dtype="float64").replace(0, np.nan)
    return (num / den).replace([np.inf, -np.inf], np.nan)


def add_onset_features(df, value_col, prefix, grp_key):
    """Track when an event (charging, queueing) first occurs in scenario."""
    if value_col not in df.columns:
        return []
    positive = df[value_col].fillna(0).gt(0).astype(bool)
    t = df["timestep"].where(positive)
    first = t.groupby(grp_key).transform(lambda s: s.ffill().cummin())
    prev = positive.groupby(grp_key).shift(1, fill_value=False).astype(bool)

    df[f"{prefix}_ever_started"] = first.notna().astype(np.int8)
    df[f"{prefix}_start_idx"] = first.fillna(-1).astype(np.int16)
    df[f"{prefix}_started_now"] = (positive & ~prev).astype(np.int8)
    df[f"{prefix}_started_early"] = (first <= 5).fillna(False).astype(np.int8)
    df[f"{prefix}_steps_since"] = np.where(
        first.notna(), (df["timestep"] - first).astype(float), -1.0
    ).astype(np.float32)
    return [
        f"{prefix}_ever_started", f"{prefix}_start_idx",
        f"{prefix}_started_now", f"{prefix}_started_early",
        f"{prefix}_steps_since",
    ]


SEQ_COLS_EXTRA = [
    "order_inflow_15m", "unique_sku_15m", "robot_active", "robot_idle",
    "robot_charging", "battery_mean", "battery_std", "low_battery_ratio",
    "charge_queue_length", "avg_charge_wait", "congestion_score",
    "max_zone_density", "blocked_path_15m", "near_collision_15m",
    "fault_count_15m", "avg_recovery_time", "task_reassign_15m",
    "replenishment_overlap", "pack_utilization", "loading_dock_util",
    "staging_area_util", "label_print_queue",
]

BASELINE_EXPAND_COLS = [
    "avg_items_per_order", "urgent_order_ratio", "heavy_item_ratio",
    "cold_chain_ratio", "sku_concentration", "bulk_order_ratio",
    "avg_trip_distance", "network_latency_ms", "air_quality_idx",
    "barcode_read_success_rate", "hvac_power_kw", "ambient_noise_db",
    "inventory_turnover_rate", "safety_score_monthly", "scanner_error_rate",
    "wms_response_time_ms", "backorder_ratio",
]


def add_pb_style_features(train_fe, test_fe, feature_cols):
    """Add PB-notebook-style features on top of existing pipeline."""
    train_out = train_fe.copy()
    test_out = test_fe.copy()
    new_cols = []

    for df in (train_out, test_out):
        grp_key = df[cfg.GROUP_COL]
        grp = df.groupby(cfg.GROUP_COL, sort=False)

        # --- 1. Time phase features ---
        ts = df["timestep"]
        df["time_frac"] = (ts / 24.0).astype(np.float32)
        df["time_remaining"] = (24 - ts).astype(np.int16)
        df["time_idx_sq"] = (df["time_frac"] ** 2).astype(np.float32)
        df["is_early_phase"] = (ts <= 5).astype(np.int8)
        df["is_mid_phase"] = ((ts >= 6) & (ts <= 15)).astype(np.int8)
        df["is_late_phase"] = (ts >= 16).astype(np.int8)

        # --- 2. Missing indicators ---
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])
                        and c not in {cfg.ID_COL, cfg.GROUP_COL, "layout_id", cfg.TARGET, "_is_train"}]
        missing_cols = [c for c in numeric_cols if df[c].isna().any()]
        for c in missing_cols:
            col_name = f"{c}__is_missing"
            df[col_name] = df[c].isna().astype(np.int8)
            if col_name not in new_cols:
                new_cols.append(col_name)
        df["n_missing_raw"] = df[missing_cols].isna().sum(axis=1).astype(np.int16) if missing_cols else 0
        df["missing_ratio_raw"] = (df["n_missing_raw"] / max(len(numeric_cols), 1)).astype(np.float32)

        # --- 3. Layout density features ---
        area = df.get("floor_area_sqm", pd.Series(np.nan, index=df.index)).replace(0, np.nan)
        if "floor_area_sqm" in df.columns:
            if "ceiling_height_m" in df.columns:
                df["warehouse_volume"] = (df["floor_area_sqm"] * df["ceiling_height_m"]).astype(float)
            if "intersection_count" in df.columns:
                df["intersection_density"] = safe_divide(df["intersection_count"], area).astype(np.float32)
            if "pack_station_count" in df.columns:
                df["pack_station_density"] = safe_divide(df["pack_station_count"], area).astype(np.float32)
            if "charger_count" in df.columns:
                df["charger_density"] = safe_divide(df["charger_count"], area).astype(np.float32)
            if "robot_total" in df.columns:
                df["robot_density_layout"] = safe_divide(df["robot_total"], area).astype(np.float32)
        if {"intersection_count", "aisle_width_avg"}.issubset(df.columns):
            df["movement_friction"] = safe_divide(df["intersection_count"], df["aisle_width_avg"]).astype(np.float32)
        if {"layout_compactness", "zone_dispersion"}.issubset(df.columns):
            df["compactness_x_dispersion"] = (df["layout_compactness"] * df["zone_dispersion"]).astype(float)
        if {"one_way_ratio", "intersection_count", "aisle_width_avg"}.issubset(df.columns):
            df["one_way_friction"] = (df["one_way_ratio"] * safe_divide(df["intersection_count"], df["aisle_width_avg"])).astype(float)

        # --- 4. Robot state decomposition ---
        if {"robot_active", "robot_idle", "robot_charging"}.issubset(df.columns):
            rts = df["robot_active"] + df["robot_idle"] + df["robot_charging"]
            rts_safe = rts.replace(0, np.nan)
            df["robot_total_state"] = rts
            df["robot_total_gap"] = rts - df["robot_total"]
            df["robot_active_share"] = safe_divide(df["robot_active"], rts_safe).astype(np.float32)
            df["robot_idle_share"] = safe_divide(df["robot_idle"], rts_safe).astype(np.float32)
            df["robot_charging_share"] = safe_divide(df["robot_charging"], rts_safe).astype(np.float32)
            df["charging_to_active_ratio"] = safe_divide(df["robot_charging"], df["robot_active"]).astype(np.float32)
            df["idle_to_active_ratio"] = safe_divide(df["robot_idle"], df["robot_active"]).astype(np.float32)

        # --- 5. Ratio features ---
        rt = df["robot_total"].replace(0, np.nan) if "robot_total" in df.columns else None
        cc = df["charger_count"].replace(0, np.nan) if "charger_count" in df.columns else None
        ps = df["pack_station_count"].replace(0, np.nan) if "pack_station_count" in df.columns else None
        ra = df["robot_active"].replace(0, np.nan) if "robot_active" in df.columns else None

        ratio_specs = [
            ("inflow_per_robot", "order_inflow_15m", rt),
            ("inflow_per_pack_station", "order_inflow_15m", ps),
            ("unique_sku_per_robot", "unique_sku_15m", rt),
            ("unique_sku_per_pack_station", "unique_sku_15m", ps),
            ("charging_per_charger", "robot_charging", cc),
            ("inflow_per_charger", "order_inflow_15m", cc),
            ("congestion_per_active", "congestion_score", ra),
            ("density_per_active", "max_zone_density", ra),
            ("fault_per_active", "fault_count_15m", ra),
            ("collision_per_active", "near_collision_15m", ra),
            ("blocked_per_active", "blocked_path_15m", ra),
            ("robot_active_per_intersection", "robot_active", df.get("intersection_count", pd.Series(np.nan, index=df.index)).replace(0, np.nan)),
        ]
        if "aisle_width_avg" in df.columns:
            aw = df["aisle_width_avg"].replace(0, np.nan)
            ratio_specs.append(("congestion_per_width", "congestion_score", aw))
            ratio_specs.append(("zone_density_per_width", "max_zone_density", aw))
            ratio_specs.append(("inflow_per_aisle_width", "order_inflow_15m", aw))
        if "staff_on_floor" in df.columns:
            ratio_specs.append(("inflow_per_staff", "order_inflow_15m", df["staff_on_floor"].replace(0, np.nan)))
        if "label_print_queue" in df.columns and ps is not None:
            ratio_specs.append(("label_queue_per_pack", "label_print_queue", ps))

        for name, num_col, denom in ratio_specs:
            if num_col in df.columns and denom is not None:
                df[name] = safe_divide(df[num_col], denom).astype(np.float32)

        # --- 6. Pressure interaction features ---
        if {"robot_charging", "charge_queue_length", "charger_count"}.issubset(df.columns):
            df["charge_pressure_pb"] = safe_divide(
                df["robot_charging"] + df["charge_queue_length"], cc
            ).astype(np.float32)
        if {"order_inflow_15m", "avg_package_weight_kg"}.issubset(df.columns):
            df["demand_mass"] = (df["order_inflow_15m"] * df["avg_package_weight_kg"]).astype(float)
            if rt is not None:
                df["demand_mass_per_robot"] = safe_divide(df["demand_mass"], rt).astype(np.float32)
        if {"order_inflow_15m", "avg_trip_distance"}.issubset(df.columns):
            df["trip_load"] = (df["order_inflow_15m"] * df["avg_trip_distance"]).astype(float)
            if rt is not None:
                df["trip_load_per_robot"] = safe_divide(df["trip_load"], rt).astype(np.float32)
        if {"order_inflow_15m", "unique_sku_15m"}.issubset(df.columns):
            df["complexity_load"] = (df["order_inflow_15m"] * df["unique_sku_15m"]).astype(float)
            if ps is not None:
                df["complexity_load_per_pack"] = safe_divide(df["complexity_load"], ps).astype(np.float32)
        if {"congestion_score", "low_battery_ratio"}.issubset(df.columns):
            df["congestion_x_lowbat"] = (df["congestion_score"] * df["low_battery_ratio"]).astype(float)
        if {"low_battery_ratio", "robot_active"}.issubset(df.columns):
            df["battery_pressure_pb"] = (df["low_battery_ratio"] * df["robot_active"]).astype(float)
        if {"charge_queue_length", "avg_charge_wait"}.issubset(df.columns):
            df["queue_wait_pressure"] = (df["charge_queue_length"] * df["avg_charge_wait"]).astype(float)
        if {"loading_dock_util", "pack_utilization"}.issubset(df.columns):
            df["dock_pack_pressure"] = (df["loading_dock_util"] * df["pack_utilization"]).astype(float)
        if {"staging_area_util", "pack_utilization"}.issubset(df.columns):
            df["staging_pack_pressure"] = (df["staging_area_util"] * df["pack_utilization"]).astype(float)
        if {"avg_recovery_time", "fault_count_15m"}.issubset(df.columns):
            df["recovery_x_fault"] = (df["avg_recovery_time"] * df["fault_count_15m"]).astype(float)
        if {"near_collision_15m", "blocked_path_15m"}.issubset(df.columns):
            df["collision_x_blocked"] = (df["near_collision_15m"] * df["blocked_path_15m"]).astype(float)

        # --- 7. Threshold features ---
        if "battery_mean" in df.columns:
            df["battery_below_44"] = np.clip(44.0 - df["battery_mean"], 0, None).astype(np.float32)
        if "charge_pressure_pb" in df.columns:
            df["charge_pressure_above_1_36"] = np.clip(df["charge_pressure_pb"] - 1.36, 0, None).astype(np.float32)

        # --- 8. Squared features ---
        for col in ["pack_utilization", "loading_dock_util", "staging_area_util"]:
            if col in df.columns:
                df[f"{col}_sq"] = (df[col].astype(float) ** 2).astype(np.float32)

        # --- 9. Onset features ---
        onset_new = add_onset_features(df, "robot_charging", "charging_onset", grp_key)
        onset_new += add_onset_features(df, "charge_queue_length", "queue_onset", grp_key)

        # --- 10. Rolling deviation & max (for SEQ_COLS not already covered) ---
        for col in SEQ_COLS_EXTRA:
            if col not in df.columns:
                continue
            rollmax_name = f"{col}_rollmax3"
            dev_name = f"{col}_dev_rmean3"
            if rollmax_name in df.columns:
                continue  # already exists
            lag1 = grp[col].shift(1)
            lg = lag1.groupby(grp_key)
            rmean = lg.rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
            rmax = lg.rolling(3, min_periods=1).max().reset_index(level=0, drop=True)
            df[rollmax_name] = rmax
            df[dev_name] = df[col] - rmean

        # --- 11. Baseline expanding features ---
        for col in BASELINE_EXPAND_COLS:
            if col not in df.columns:
                continue
            emean_name = f"{col}_expmean"
            delta_name = f"{col}_delta_expmean"
            if emean_name in df.columns:
                continue
            prev = grp[col].shift(1)
            emean = prev.groupby(grp_key).expanding(min_periods=1).mean().reset_index(level=0, drop=True)
            df[emean_name] = emean
            df[delta_name] = df[col] - emean

        # --- 12. Layout one-hot ---
        if "layout_type" in df.columns:
            for lt in ["grid", "hub_spoke", "hybrid", "narrow"]:
                df[f"layout_type_{lt}"] = (df["layout_type"] == lt).astype(np.int8)
        elif "layout_type_enc" in df.columns:
            for lt_val, lt_name in [(0, "grid"), (1, "hub_spoke"), (2, "hybrid"), (3, "narrow")]:
                df[f"layout_type_{lt_name}"] = (df["layout_type_enc"] == lt_val).astype(np.int8)

        # --- 13. Scenario-level aggregation (mean/std/max/min across all 25 timesteps) ---
        sc_agg_candidates = [
            "order_inflow_15m", "congestion_score", "low_battery_ratio",
            "robot_charging", "charge_queue_length", "robot_active", "robot_idle",
            "battery_mean", "pack_utilization", "loading_dock_util",
            "staging_area_util", "fault_count_15m", "near_collision_15m",
            "blocked_path_15m", "avg_charge_wait", "avg_trip_distance",
            "charge_pressure_pb", "battery_pressure_pb", "demand_mass_per_robot",
            "congestion_x_lowbat", "queue_wait_pressure",
        ]
        sc_agg_cols = [c for c in sc_agg_candidates if c in df.columns]
        sc_grp = df.groupby(cfg.GROUP_COL, sort=False)[sc_agg_cols]
        sc_mean = sc_grp.transform("mean").add_suffix("__sc_mean")
        sc_std  = sc_grp.transform("std").fillna(0).add_suffix("__sc_std")
        sc_max  = sc_grp.transform("max").add_suffix("__sc_max")
        sc_min  = sc_grp.transform("min").add_suffix("__sc_min")
        for agg_df in (sc_mean, sc_std, sc_max, sc_min):
            for c in agg_df.columns:
                df[c] = agg_df[c].astype(np.float32)

    # Collect all new columns (union from both dfs)
    base_set = set(feature_cols)
    all_cols_train = set(train_out.columns)
    all_cols_test = set(test_out.columns)
    exclude = {cfg.ID_COL, cfg.GROUP_COL, "layout_id", cfg.TARGET, "_is_train", "layout_type"}
    new_feature_cols = sorted(
        (all_cols_train & all_cols_test) - base_set - exclude
    )
    final_cols = list(feature_cols) + [c for c in new_feature_cols if c not in base_set]

    log(f"  PB-style features: {len(feature_cols)} -> {len(final_cols)} (+{len(final_cols) - len(feature_cols)})")
    return train_out, test_out, final_cols


# ── Sample weighting ────────────────────────────────────────────────

def build_v8_sample_weight(y_raw, time_idx=None):
    """Train-only sample weight: boost high-delay and late-timestep samples."""
    w = np.ones(len(y_raw), dtype=np.float32)
    q90 = np.nanquantile(y_raw, 0.90)
    q95 = np.nanquantile(y_raw, 0.95)
    q99 = np.nanquantile(y_raw, 0.99)
    w += 0.15 * (y_raw >= q90).astype(np.float32)
    w += 0.30 * (y_raw >= q95).astype(np.float32)
    w += 0.60 * (y_raw >= q99).astype(np.float32)
    if time_idx is not None:
        t = np.asarray(time_idx, dtype=np.float32)
        t_max = max(t.max(), 1.0)
        w += 0.08 * (t / t_max)
    w /= w.mean()  # normalize to mean 1.0
    log(f"  sample weight: q90={q90:.1f} q95={q95:.1f} q99={q99:.1f} w_range=[{w.min():.3f}, {w.max():.3f}]")
    return w


# ── Per-model training ──────────────────────────────────────────────

MODEL_SPECS = [
    {
        "name": "lgb_mae_raw",
        "family": "lgb",
        "transform": "none",
        "params": {
            "objective": "mae", "metric": "mae",
            "n_estimators": 3000, "learning_rate": 0.03,
            "num_leaves": 96, "max_depth": -1,
            "min_child_samples": 80,
            "subsample": 0.9, "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.1, "reg_lambda": 1.5,
            "verbosity": -1,
        },
    },
    {
        "name": "lgb_huber_log",
        "family": "lgb",
        "transform": "log1p",
        "params": {
            "objective": "huber", "alpha": 0.9,
            "metric": "mae",
            "n_estimators": 3000, "learning_rate": 0.03,
            "num_leaves": 128, "max_depth": -1,
            "min_child_samples": 60,
            "subsample": 0.9, "subsample_freq": 1,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.05, "reg_lambda": 1.0,
            "verbosity": -1,
        },
    },
    {
        "name": "xgb_mae_raw",
        "family": "xgb",
        "transform": "none",
        "params": {
            "objective": "reg:absoluteerror", "eval_metric": "mae",
            "n_estimators": 3000, "learning_rate": 0.03,
            "max_depth": 8, "min_child_weight": 6,
            "subsample": 0.9, "colsample_bytree": 0.85,
            "reg_alpha": 0.05, "reg_lambda": 1.5,
            "tree_method": "hist", "verbosity": 0,
        },
    },
    {
        "name": "cat_mae_log",
        "family": "cat",
        "transform": "log1p",
        "params": {
            "loss_function": "MAE", "eval_metric": "MAE",
            "iterations": 3000, "learning_rate": 0.03,
            "depth": 8, "l2_leaf_reg": 5.0,
            "subsample": 0.9,
            "verbose": 100,
            "early_stopping_rounds": 100,
        },
    },
    {
        "name": "lgb_mae_log",
        "family": "lgb",
        "transform": "log1p",
        "params": {
            "objective": "mae", "metric": "mae",
            "n_estimators": 3000, "learning_rate": 0.03,
            "max_depth": 8, "num_leaves": 127,
            "min_child_samples": 50,
            "subsample": 0.7, "colsample_bytree": 0.7,
            "reg_alpha": 0.5, "reg_lambda": 1.0,
            "verbosity": -1,
        },
    },
]


def train_single_model_cv(spec, X, y_raw, groups, X_test,
                          n_folds=5, early_stop=100, seed=42,
                          sample_weight=None, ckpt_prefix="v8"):
    """Train one model type with its own objective and transform."""
    name = spec["name"]
    family = spec["family"]
    transform = spec["transform"]
    params = spec["params"].copy()
    ckpt_name = f"{ckpt_prefix}_{name}_seed{seed}"

    cached = v5.load_ckpt(ckpt_name)
    if cached is not None:
        return cached

    # Set seed
    if family == "lgb":
        params["random_state"] = seed
    elif family == "xgb":
        params["random_state"] = seed
    elif family == "cat":
        params["random_seed"] = seed

    y_tr_all = v5.transform_target(y_raw, transform)
    gkf = GroupKFold(n_splits=n_folds)
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        fold_ckpt = f"{ckpt_name}_fold{fold}"
        fc = v5.load_ckpt(fold_ckpt)
        if fc is not None:
            oof[val_idx] = fc["oof"]
            test_pred += fc["test"] / n_folds
            log(f"  {name} seed{seed} fold{fold+1} MAE: {fc['mae']:.4f} (캐시)")
            continue

        ft0 = time.time()
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_tr_all[tr_idx], y_tr_all[val_idx]
        y_val_raw = y_raw[val_idx]
        sw = sample_weight[tr_idx] if sample_weight is not None else None

        if family == "lgb":
            m = v5.train_lgb(X_tr, y_tr, X_val, y_val, params, early_stop, sw)
        elif family == "xgb":
            m = v5.train_xgb(X_tr, y_tr, X_val, y_val, params, early_stop, sw)
        elif family == "cat":
            m = v5.train_cat(X_tr, y_tr, X_val, y_val, params, sw)
        else:
            raise ValueError(f"Unknown family: {family}")

        p_val = v5.inverse_transform(m.predict(X_val), transform)
        p_test = v5.inverse_transform(m.predict(X_test), transform)
        oof[val_idx] = p_val
        test_pred += p_test / n_folds
        fold_mae = mean_absolute_error(y_val_raw, p_val)
        log(f"  {name} seed{seed} fold{fold+1} MAE: {fold_mae:.4f} ({time.time()-ft0:.0f}s)")

        v5.save_ckpt(fold_ckpt, {"oof": p_val, "test": p_test, "mae": fold_mae})

    cv = mean_absolute_error(y_raw, oof)
    result = (cv, test_pred, oof)
    v5.save_ckpt(ckpt_name, result)
    log(f"  {name} seed{seed} CV: {cv:.4f}")
    return result


def train_model_multiseed(spec, X, y_raw, groups, X_test,
                          seeds=(42, 123, 2026), ckpt_prefix="v8", **kwargs):
    """Multi-seed wrapper for a single model spec."""
    all_oof, all_test = [], []
    for seed in seeds:
        cv, test, oof = train_single_model_cv(
            spec, X, y_raw, groups, X_test, seed=seed,
            ckpt_prefix=ckpt_prefix, **kwargs,
        )
        all_oof.append(oof)
        all_test.append(test)

    oof_avg = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_avg = mean_absolute_error(y_raw, oof_avg)
    log(f"  {spec['name']} {len(seeds)}-seed avg CV: {cv_avg:.4f}")
    return cv_avg, test_avg, oof_avg


# ── Submission helpers ──────────────────────────────────────────────

def save_submission(test_fe, pred, name):
    sub = pd.DataFrame({
        cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
        cfg.TARGET: np.clip(pred, 0, None),
    })
    sample = pd.read_csv(cfg.SAMPLE_SUB_FILE)
    assert len(sub) == len(sample), f"row mismatch: {len(sub)} != {len(sample)}"
    assert sub[cfg.TARGET].isna().sum() == 0, "NaN"
    assert (sub[cfg.TARGET] >= 0).all(), "negative"
    path = os.path.join(cfg.SUBMISSION_DIR, f"{name}_submission.csv")
    sub.to_csv(path, index=False, float_format="%.10f", lineterminator="\n")
    log(f"  저장: {path} mean={sub[cfg.TARGET].mean():.3f} std={sub[cfg.TARGET].std():.3f} max={sub[cfg.TARGET].max():.3f}")
    return path


# ── Main orchestrator ───────────────────────────────────────────────

def run_v8(sanity=False, ckpt_prefix="v8"):
    log("=" * 60)
    log(f"v8 — PB features + objective diversity + sample weight  [prefix={ckpt_prefix}]")
    if sanity:
        log("!! SANITY CHECK 모드")
    log("=" * 60)

    # 1. Build strict base features
    train_fe, test_fe, feature_cols = strict.build_strict_data(sanity=sanity)

    # 2. Add pressure + lead/forward (same as attack3: expanded + core)
    log("  attack3-style features (expanded pressure + core lead)...")
    train_fe, test_fe, feature_cols = attack.add_lead_features_split(
        train_fe, test_fe, feature_cols,
        lead_mode="core", base_mode="expanded",
    )

    # 3. Add PB-style features
    log("  PB-style feature augmentation...")
    train_fe, test_fe, feature_cols = add_pb_style_features(
        train_fe, test_fe, feature_cols,
    )

    X = train_fe[feature_cols]
    X_test = strict.align_test_features(test_fe, feature_cols)
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values

    log(f"  Final features: {len(feature_cols)}, train={X.shape}, test={X_test.shape}")

    # 4. Sample weight
    time_idx = train_fe["timestep"].values if "timestep" in train_fe.columns else None
    sw = build_v8_sample_weight(y_raw, time_idx)

    # 5. Train diverse models
    seeds = [42] if sanity else [42, 123, 2026]
    n_folds = 2 if sanity else 5
    early_stop = 30 if sanity else 100

    specs = MODEL_SPECS
    if sanity:
        # Reduce iterations for sanity
        specs = []
        for s in MODEL_SPECS:
            s2 = {**s, "params": s["params"].copy()}
            if s2["family"] in ("lgb", "xgb"):
                s2["params"]["n_estimators"] = 200
            elif s2["family"] == "cat":
                s2["params"]["iterations"] = 200
            specs.append(s2)

    model_results = {}
    for spec in specs:
        log(f"\n  === {spec['name']} ({spec['family']}, {spec['transform']}) ===")
        cv, test, oof = train_model_multiseed(
            spec, X, y_raw, groups, X_test,
            seeds=seeds, n_folds=n_folds, early_stop=early_stop,
            sample_weight=sw, ckpt_prefix=ckpt_prefix,
        )
        model_results[spec["name"]] = {"cv": cv, "test": test, "oof": oof}

    # 6. Load existing checkpoints
    log("\n  === Existing checkpoints ===")
    existing = {}
    for ckpt_name, ckpt_dir, prefix in [
        ("attack3_gbdt", attack.ATTACK_CKPT_DIR, "attack_3_expanded_core_lead_gbdt"),
        ("attack4_gbdt", attack.ATTACK_CKPT_DIR, "attack_4_expanded_ops_lead_gbdt"),
        ("strict2_tabnet", strict.STRICT_CKPT_DIR, "strict_2_focused_tabnet"),
    ]:
        try:
            cvs, test_pred, oof_pred = attack.load_strict_seed_bundle(prefix, seeds=(42, 123, 2026))
            if len(oof_pred) != len(y_raw):
                log(f"  {ckpt_name}: size mismatch ({len(oof_pred)} vs {len(y_raw)}), skipping")
                continue
            cv = mean_absolute_error(y_raw, oof_pred)
            existing[ckpt_name] = {"cv": cv, "test": test_pred, "oof": oof_pred}
            log(f"  {ckpt_name}: CV={cv:.4f}")
        except FileNotFoundError:
            log(f"  {ckpt_name}: not found, skipping")

    # 7. Build blend candidates
    all_names = []
    all_oofs = []
    all_tests = []

    for name, res in model_results.items():
        all_names.append(name)
        all_oofs.append(res["oof"])
        all_tests.append(res["test"])

    for name, res in existing.items():
        all_names.append(name)
        all_oofs.append(res["oof"])
        all_tests.append(res["test"])

    # 8. N-way blend
    log("\n  === N-way blend optimization ===")
    weights, blend_cv = v5.optimize_blend_multi(all_oofs, y_raw, all_names)

    oof_blend = sum(w * o for w, o in zip(weights, all_oofs))
    test_blend = sum(w * t for w, t in zip(weights, all_tests))

    # 9. Save submission variants
    log("\n  === Submission variants ===")

    # Variant 1: full blend noclip
    save_submission(test_fe, test_blend, "v8_diverse_blend_noclip")

    # Variant 2: full blend clip130
    clip130 = np.percentile(oof_blend, 99) * 1.30
    save_submission(test_fe, np.clip(test_blend, 0, clip130), "v8_diverse_blend_clip130")
    cv_clip130 = mean_absolute_error(y_raw, np.clip(oof_blend, 0, clip130))
    log(f"  clip130={clip130:.2f}, CV clipped={cv_clip130:.4f}")

    # Variant 3: v8 new models only
    v8_names = list(model_results.keys())
    v8_oofs = [model_results[n]["oof"] for n in v8_names]
    v8_tests = [model_results[n]["test"] for n in v8_names]
    w_v8, cv_v8 = v5.optimize_blend_multi(v8_oofs, y_raw, v8_names)
    test_v8_only = sum(w * t for w, t in zip(w_v8, v8_tests))
    save_submission(test_fe, test_v8_only, "v8_new_models_only")

    # Variant 4: top3 by CV
    sorted_models = sorted(model_results.items(), key=lambda x: x[1]["cv"])
    top3 = sorted_models[:3]
    top3_names = [n for n, _ in top3]
    top3_oofs = [model_results[n]["oof"] for n in top3_names]
    top3_tests = [model_results[n]["test"] for n in top3_names]
    w_top3, cv_top3 = v5.optimize_blend_multi(top3_oofs, y_raw, top3_names)
    test_top3 = sum(w * t for w, t in zip(w_top3, top3_tests))
    save_submission(test_fe, test_top3, "v8_top3_blend")

    # Variant 5: v8 blend + attack3 noclip submission average
    if "attack3_gbdt" in existing:
        hybrid = 0.5 * test_blend + 0.5 * existing["attack3_gbdt"]["test"]
        save_submission(test_fe, hybrid, "v8_attack_hybrid")

    # Variant 6: conservative clip110
    clip110 = np.percentile(oof_blend, 99) * 1.10
    save_submission(test_fe, np.clip(test_blend, 0, clip110), "v8_conservative_clip110")

    # Summary
    log("\n" + "=" * 60)
    log("v8 결과 요약")
    log("=" * 60)
    log(f"  Features: {len(feature_cols)}")
    for name, res in model_results.items():
        log(f"  {name:<20} CV={res['cv']:.4f}")
    for name, res in existing.items():
        log(f"  {name:<20} CV={res['cv']:.4f} (cached)")
    log(f"  N-way blend CV: {blend_cv:.4f}")
    log(f"  clip130 CV: {cv_clip130:.4f}")
    log(f"  v8-only blend CV: {cv_v8:.4f}")
    log(f"  top3 blend CV: {cv_top3:.4f}")
    log("=" * 60)


def main():
    parser = argparse.ArgumentParser(description="v8 diverse pipeline")
    parser.add_argument("--sanity", action="store_true")
    parser.add_argument("--ckpt_prefix", default="v8", help="체크포인트 prefix (기본: v8)")
    args = parser.parse_args()

    ckpt_dir = V8_SANITY_CKPT_DIR if args.sanity else V8_CKPT_DIR
    v5.CKPT_DIR = ckpt_dir
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)

    t0 = time.time()
    run_v8(sanity=args.sanity, ckpt_prefix=args.ckpt_prefix)
    log(f"\n총 소요: {(time.time() - t0) / 60:.1f}분")


if __name__ == "__main__":
    main()


---
## run_v8_tabnet.py (TabNet + v8 피처 빌드)


In [ ]:
# run_v8_tabnet.py
# ============================================================
# 아래 코드를 run_v8_tabnet.py 파일로 저장하세요

"""
File: run_v8_tabnet.py
"""


In [ ]:
"""
v8 TabNet — v8 714 피처로 TabNet 학습 후 v8 GBDT blend에 추가.
"""
import argparse
import os
import time

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler

import config as cfg
import run_experiments_v5 as v5
import run_experiments_strict as strict
import run_experiments_attack as attack
import run_experiments_v8 as v8

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
V8_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8")
V8_TABNET_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8_tabnet")


def log(msg):
    v5.log(msg)


def build_v8_features(sanity=False):
    log("  strict base features 빌드...")
    train_fe, test_fe, feature_cols = strict.build_strict_data(sanity=sanity)

    log("  attack3-style lead features 추가...")
    train_fe, test_fe, feature_cols = attack.add_lead_features_split(
        train_fe, test_fe, feature_cols,
        lead_mode="core", base_mode="expanded",
    )

    log("  PB-style features 추가...")
    train_fe, test_fe, feature_cols = v8.add_pb_style_features(
        train_fe, test_fe, feature_cols,
    )

    X_df = train_fe[feature_cols]
    X_test_df = strict.align_test_features(test_fe, feature_cols)
    y_raw = train_fe[cfg.TARGET].values
    groups = train_fe[cfg.GROUP_COL].values

    # StandardScaler fit on train only (strict-clean), fillna(0) before scaling
    scaler = StandardScaler()
    X = scaler.fit_transform(X_df.fillna(0)).astype(np.float32)
    X_test = scaler.transform(X_test_df.fillna(0)).astype(np.float32)
    log(f"  StandardScaler 적용 완료")

    log(f"  피처: {len(feature_cols)}, train={X.shape}, test={X_test.shape}")
    return X, X_test, y_raw, groups, test_fe


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--sanity", action="store_true")
    args = parser.parse_args()

    ckpt_dir = V8_TABNET_CKPT_DIR
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)

    v5.CKPT_DIR = ckpt_dir

    log("=" * 60)
    log("v8 TabNet — 714 피처 TabNet 학습")
    if args.sanity:
        log("!! SANITY CHECK 모드")
    log("=" * 60)

    t0 = time.time()

    # 1. v8 피처 빌드
    X, X_test, y_raw, groups, test_fe = build_v8_features(sanity=args.sanity)

    # 2. TabNet 학습 (3 seed)
    seeds = [42] if args.sanity else [42, 123, 2026]
    n_folds = 2 if args.sanity else 5
    max_epochs = 50 if args.sanity else 200

    log(f"\n  === TabNet (n_d=32, seeds={seeds}, folds={n_folds}) ===")
    cv_tab, test_tab, oof_tab = v5.run_tabnet_multiseed(
        X, y_raw, groups, X_test,
        seeds=seeds,
        ckpt_prefix="v8_tabnet",
        n_folds=n_folds,
        max_epochs=max_epochs,
    )
    log(f"  v8 TabNet 3-seed avg CV: {cv_tab:.4f}")

    # 3. v8 GBDT OOF 로드해서 함께 blend (full 모드만)
    if args.sanity:
        log("\n  sanity 모드: 블렌드 스킵 (OOF 크기 불일치)")
        log(f"  v8 TabNet CV: {cv_tab:.4f}")
        log(f"  총 소요: {(time.time() - t0) / 60:.1f}분")
        return

    log("\n  === v8 GBDT + v8 TabNet 블렌드 ===")
    v5.CKPT_DIR = V8_CKPT_DIR
    model_oofs, model_tests, model_names = [], [], []

    for name in ["lgb_huber_log", "lgb_mae_log", "cat_mae_log", "lgb_mae_raw", "xgb_mae_raw"]:
        oofs, tests = [], []
        for seed in [42, 123, 2026]:
            ck = v5.load_ckpt(f"v8_{name}_seed{seed}")
            if ck is not None:
                _, tp, op = ck
                oofs.append(op); tests.append(tp)
        if oofs:
            model_oofs.append(np.mean(oofs, axis=0))
            model_tests.append(np.mean(tests, axis=0))
            model_names.append(name)

    # strict2 TabNet (198 피처)
    v5.CKPT_DIR = strict.STRICT_CKPT_DIR
    tab_oofs = []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"strict_2_focused_tabnet_seed{seed}")
        if ck is not None:
            _, _, op = ck
            tab_oofs.append(op)
    if tab_oofs:
        model_oofs.append(np.mean(tab_oofs, axis=0))
        model_tests.append(np.mean([v5.load_ckpt(f"strict_2_focused_tabnet_seed{s}")[1]
                                    for s in [42, 123, 2026]], axis=0))
        model_names.append("strict2_tabnet")

    # v8 TabNet (714 피처) 추가
    model_oofs.append(oof_tab)
    model_tests.append(test_tab)
    model_names.append("v8_tabnet")

    weights, blend_cv = v5.optimize_blend_multi(model_oofs, y_raw, model_names)
    log(f"  {len(model_names)}-way blend CV: {blend_cv:.4f}")
    for n, w in zip(model_names, weights):
        log(f"    {n}: {w:.3f}")

    # 4. 제출 파일 생성
    oof_blend = sum(w * o for w, o in zip(weights, model_oofs))
    test_blend = sum(w * t for w, t in zip(weights, model_tests))

    # noclip
    sub = pd.DataFrame({
        cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
        cfg.TARGET: np.clip(test_blend, 0, None),
    })
    path_noclip = os.path.join(cfg.SUBMISSION_DIR, "v8_tabnet_blend_noclip_submission.csv")
    sub.to_csv(path_noclip, index=False, float_format="%.10f", lineterminator="\n")
    log(f"  저장: {path_noclip}  mean={sub[cfg.TARGET].mean():.3f} std={sub[cfg.TARGET].std():.3f} max={sub[cfg.TARGET].max():.3f}")

    # clip130
    clip_th = np.percentile(oof_blend, 99) * 1.30
    sub_clip = sub.copy()
    sub_clip[cfg.TARGET] = np.clip(test_blend, 0, clip_th)
    path_clip = os.path.join(cfg.SUBMISSION_DIR, "v8_tabnet_blend_clip130_submission.csv")
    sub_clip.to_csv(path_clip, index=False, float_format="%.10f", lineterminator="\n")
    cv_clip = mean_absolute_error(y_raw, np.clip(oof_blend, 0, clip_th))
    log(f"  저장: {path_clip}  clip_th={clip_th:.2f} CV_clip={cv_clip:.4f}")

    log("\n" + "=" * 60)
    log("결과 요약")
    log("=" * 60)
    log(f"  v8 TabNet CV: {cv_tab:.4f}")
    log(f"  blend noclip CV: {blend_cv:.4f}")
    log(f"  blend clip130 CV: {cv_clip:.4f}")
    log(f"  총 소요: {(time.time() - t0) / 60:.1f}분")
    log("=" * 60)


if __name__ == "__main__":
    main()


---
## run_v8_transformer.py (Transformer v4 / MLP 학습)


In [ ]:
# run_v8_transformer.py
# ============================================================
# 아래 코드를 run_v8_transformer.py 파일로 저장하세요

"""
File: run_v8_transformer.py
"""


In [ ]:
"""
v8 Strong Transformer — 25 timestep 시퀀스 모델.
(batch, 25, 714) 입력, pre-norm Transformer Encoder, learnable positional embedding.
"""
import argparse, math, os, time
from contextlib import nullcontext

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

import config as cfg
import run_experiments_v5 as v5
import run_experiments_strict as strict
from sklearn.preprocessing import StandardScaler
from run_v8_tabnet import build_v8_features

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
V8_CKPT_DIR    = os.path.join(BASE_DIR, "checkpoints_v8")
V8_TAB_CKPT    = os.path.join(BASE_DIR, "checkpoints_v8_tabnet")
V8_TF_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8_transformer")


def build_base_features(sanity=False):
    """198 base features only — lag/rolling 없이 시퀀스 모델이 직접 시간 패턴 학습"""
    train_fe, test_fe, feature_cols = strict.build_strict_data(sanity=sanity)
    X_df      = train_fe[feature_cols]
    X_test_df = strict.align_test_features(test_fe, feature_cols)
    y_raw     = train_fe[cfg.TARGET].values
    groups    = train_fe[cfg.GROUP_COL].values
    scaler    = StandardScaler()
    X         = scaler.fit_transform(X_df.fillna(0)).astype(np.float32)
    X_test    = scaler.transform(X_test_df.fillna(0)).astype(np.float32)
    v5.log(f"  base features: {len(feature_cols)}, train={X.shape}, test={X_test.shape}")
    return X, X_test, y_raw, groups, test_fe
N_TS = 25


# ── Models ────────────────────────────────────────────────────────────
class Conv1DNet(nn.Module):
    """Multi-scale 1D CNN — 인접 timestep 로컬 패턴 학습 (kernel 3+5)"""
    def __init__(self, n_feat, channels=128, dropout=0.2, **_):
        super().__init__()
        self.proj  = nn.Linear(n_feat, channels)
        self.conv3 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.norm1 = nn.LayerNorm(channels)
        self.norm2 = nn.LayerNorm(channels)
        self.drop  = nn.Dropout(dropout)
        self.head  = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(channels // 2, 1),
        )

    def forward(self, x):              # x: (B, 25, n_feat)
        h = self.proj(x)               # (B, 25, C)
        ht = h.transpose(1, 2)         # (B, C, 25)
        h = self.norm1(h + self.conv3(ht).transpose(1, 2))
        h = self.norm2(h + self.conv5(h.transpose(1, 2)).transpose(1, 2))
        return self.head(self.drop(h)).squeeze(-1)  # (B, 25)


class BiLSTMAttention(nn.Module):
    def __init__(self, n_feat, hidden=256, n_layers=2, n_heads=8, dropout=0.2, **_):
        super().__init__()
        d = hidden * 2  # bidirectional output dim
        self.proj = nn.Linear(n_feat, hidden)
        self.lstm = nn.LSTM(hidden, hidden, num_layers=n_layers,
                            batch_first=True, bidirectional=True, dropout=dropout if n_layers > 1 else 0)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ffn   = nn.Sequential(nn.Linear(d, d * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d * 2, d))
        self.norm2 = nn.LayerNorm(d)
        self.head  = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, 1))

    def forward(self, x):              # x: (B, 25, n_feat)
        h, _ = self.lstm(self.proj(x)) # (B, 25, hidden*2)
        a, _ = self.attn(h, h, h)
        h = self.norm1(h + a)
        h = self.norm2(h + self.ffn(h))
        return self.head(h).squeeze(-1)  # (B, 25)


class StrongTransformer(nn.Module):
    def __init__(self, n_feat, d_model=256, n_heads=8, n_layers=6, d_ff=1024, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(n_feat, d_model)
        self.pos  = nn.Embedding(N_TS, d_model)
        enc = nn.TransformerEncoderLayer(
            d_model, n_heads, d_ff, dropout,
            batch_first=True, norm_first=True)   # pre-LayerNorm
        self.enc  = nn.TransformerEncoder(enc, n_layers)
        # 2-layer MLP head with GELU
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x):          # x: (B, 25, n_feat)
        pos = torch.arange(x.shape[1], device=x.device)
        h0 = self.proj(x) + self.pos(pos)
        return self.head(self.enc(h0) + h0).squeeze(-1)  # skip conn + (B, 25)


# ── Data helpers ───────────────────────────────────────────────────────
def flat_to_seq(X_flat, y_flat, groups):
    """row-level → (n_sc, N_TS, n_feat), (n_sc, N_TS)|None, unique_sc"""
    unique_sc, sc_to_i = [], {}
    for sc in groups:
        if sc not in sc_to_i:
            sc_to_i[sc] = len(unique_sc)
            unique_sc.append(sc)
    n_sc = len(unique_sc)
    X_seq = np.zeros((n_sc, N_TS, X_flat.shape[1]), dtype=np.float32)
    y_seq = np.zeros((n_sc, N_TS), dtype=np.float32) if y_flat is not None else None
    cnt = np.zeros(n_sc, dtype=np.int32)
    for r, sc in enumerate(groups):
        i = sc_to_i[sc]
        X_seq[i, cnt[i]] = X_flat[r]
        if y_seq is not None:
            y_seq[i, cnt[i]] = y_flat[r]
        cnt[i] += 1
    return X_seq, y_seq, np.array(unique_sc)


def seq_to_flat(pred_seq, groups, unique_sc):
    """(n_sc, N_TS) → row-level (N,) in original row order"""
    sc_to_i = {sc: i for i, sc in enumerate(unique_sc)}
    out, cnt = np.zeros(len(groups)), {}
    for r, sc in enumerate(groups):
        p = cnt.get(sc, 0)
        out[r] = pred_seq[sc_to_i[sc], p]
        cnt[sc] = p + 1
    return out


# ── CV training ────────────────────────────────────────────────────────
def run_transformer_cv_seq(X_seq, y_seq, X_test_seq, scenario_ids,
                            model_cls=StrongTransformer, model_kwargs=None,
                            n_folds=5, base_seed=42, max_epochs=200,
                            batch_size=512, patience=30, peak_lr=3e-4, warmup=10,
                            ckpt_name=None, loss_fn=None):
    if ckpt_name:
        cached = v5.load_ckpt(ckpt_name)
        if cached is not None:
            cv_c, oof_c, test_c = cached
            if oof_c.shape[0] == len(X_seq) and test_c.shape[0] == len(X_test_seq):
                return cached
            v5.log(f"  {ckpt_name}: 크기 불일치 ({oof_c.shape[0]} vs {len(X_seq)}), 재학습")

    if model_kwargs is None:
        model_kwargs = {}
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_sc, _, n_feat = X_seq.shape
    use_amp = device.type == "cuda"
    v5.log(f"  {model_cls.__name__}  device:{device}  kwargs={model_kwargs}  seed={base_seed}")
    torch.manual_seed(base_seed)

    gkf       = GroupKFold(n_splits=n_folds)
    oof_seq   = np.zeros((n_sc, N_TS))
    test_seq  = np.zeros((len(X_test_seq), N_TS))
    fold_pfx  = f"{ckpt_name}_fold" if ckpt_name else None
    X_test_t  = torch.from_numpy(X_test_seq).to(device)
    amp       = torch.autocast("cuda", dtype=torch.bfloat16) if use_amp else nullcontext()

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(np.arange(n_sc), groups=scenario_ids)):
        if fold_pfx:
            fc = v5.load_ckpt(f"{fold_pfx}_{fold}")
            if fc is not None:
                if fc["oof"].shape[0] == len(val_idx):
                    oof_seq[val_idx] = fc["oof"]
                    test_seq += fc["test"] / n_folds
                    v5.log(f"  Fold {fold+1} MAE: {fc['mae']:.4f}  (캐시)")
                    continue
                v5.log(f"  Fold {fold+1} 캐시 크기 불일치, 재학습")

        ft0     = time.time()
        X_tr    = torch.from_numpy(X_seq[tr_idx]).to(device)
        y_tr    = torch.from_numpy(np.log1p(y_seq[tr_idx]).astype(np.float32)).to(device)
        X_val_t = torch.from_numpy(X_seq[val_idx]).to(device)
        y_val_r = y_seq[val_idx]

        model = model_cls(n_feat, **model_kwargs).to(device)
        wd    = model_kwargs.get("weight_decay", 1e-4)
        opt   = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=wd)
        def lr_lambda(ep):
            if ep < warmup:
                return ep / max(warmup, 1)
            prog = (ep - warmup) / max(max_epochs - warmup, 1)
            return 0.5 * (1 + math.cos(math.pi * prog))
        sched   = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        loss_fn = nn.HuberLoss(delta=0.5)

        best_mae, best_state, p_cnt = float("inf"), None, 0
        _loss_fn = loss_fn if loss_fn is not None else nn.HuberLoss(delta=0.5)

        for epoch in range(max_epochs):
            model.train()
            perm = torch.randperm(len(X_tr), device=device)
            for i in range(0, len(X_tr), batch_size):
                xb, yb = X_tr[perm[i:i+batch_size]], y_tr[perm[i:i+batch_size]]
                with amp:
                    loss = _loss_fn(model(xb), yb)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()

            if (epoch + 1) % 5 == 0:
                model.eval()
                with torch.no_grad(), amp:
                    vp = np.expm1(model(X_val_t).cpu().float().numpy()).clip(0)
                vm = mean_absolute_error(y_val_r.flatten(), vp.flatten())
                if vm < best_mae:
                    best_mae, p_cnt = vm, 0
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    p_cnt += 1
                if p_cnt >= patience // 5:
                    v5.log(f"    early stop ep={epoch+1}  best={best_mae:.4f}")
                    break

        if best_state:
            model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad(), amp:
            oof_pred  = np.expm1(model(X_val_t).cpu().float().numpy()).clip(0)
            test_pred = np.expm1(model(X_test_t).cpu().float().numpy()).clip(0)

        oof_seq[val_idx]  = oof_pred
        test_seq         += test_pred / n_folds
        fold_mae = mean_absolute_error(y_val_r.flatten(), oof_pred.flatten())
        v5.log(f"  Fold {fold+1} MAE: {fold_mae:.4f}  ({time.time()-ft0:.0f}s)")
        if fold_pfx:
            v5.save_ckpt(f"{fold_pfx}_{fold}", {"oof": oof_pred, "test": test_pred, "mae": fold_mae})

    cv = mean_absolute_error(y_seq.flatten(), oof_seq.flatten())
    v5.log(f"  Transformer CV: {cv:.4f}")
    result = (cv, oof_seq, test_seq)
    if ckpt_name:
        v5.save_ckpt(ckpt_name, result)
    return result


# ── main ───────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--sanity", action="store_true")
    parser.add_argument("--v2", action="store_true", help="warmup+cosine, 6layers, patience30, 200ep")
    parser.add_argument("--v3", action="store_true", help="v2 + dropout0.3, wd5e-4, lr1e-4, patience50, 300ep")
    parser.add_argument("--v4", action="store_true", help="v2 구조 + lr=2e-4, patience35, ep250, L1Loss")
    parser.add_argument("--v5", action="store_true", help="소형: d=128, L=4, lr=3e-4, L1Loss")
    parser.add_argument("--cnn", action="store_true", help="1D CNN: channels=128, kernel 3+5, L1Loss")
    parser.add_argument("--lstm", action="store_true", help="BiLSTM+Attention, hidden=256, 2layers")
    parser.add_argument("--base", action="store_true", help="198 base features only (lag없는 원시 피처 → LSTM이 시간패턴 학습)")
    args = parser.parse_args()

    os.makedirs(V8_TF_CKPT_DIR, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)
    v5.CKPT_DIR = V8_TF_CKPT_DIR

    tag = "base_lstm" if args.base else ("lstm" if args.lstm else ("cnn" if args.cnn else ("v5" if args.v5 else ("v4" if args.v4 else ("v3" if args.v3 else ("v2" if args.v2 else "v1"))))))
    v5.log("=" * 60)
    v5.log(f"v8 Sequence Model [{tag}]")
    if args.sanity: v5.log("!! SANITY 모드")
    if args.base: v5.log("  Base 198 피처 BiLSTM+Attention (lag/rolling 없음 — 시퀀스가 직접 학습)")
    elif args.lstm: v5.log("  BiLSTM+Attention: hidden=256 layers=2 heads=8 / lr=3e-4 warmup=10 patience=30")
    elif args.cnn: v5.log("  1D CNN: channels=128 kernel=3+5 / lr=3e-4 warmup=10 dropout=0.2 patience=30 L1Loss")
    elif args.v5: v5.log("  v5: d=128 L=4 heads=4 / lr=3e-4 warmup=10 dropout=0.2 patience=30 ep=200 L1Loss")
    elif args.v4: v5.log("  v4: lr=2e-4 / warmup=10 / dropout=0.2 / patience=35 / ep=250 / L1Loss")
    elif args.v3: v5.log("  v3: lr=1e-4 / warmup=20 / dropout=0.3 / wd=5e-4 / patience=50 / ep=300")
    elif args.v2: v5.log("  v2: lr=3e-4 / warmup=10 / dropout=0.2 / patience=30 / ep=200")
    v5.log("=" * 60)
    t0 = time.time()

    # 1. 피처 빌드
    if args.base:
        X_flat, X_test_flat, y_raw, groups, test_fe = build_base_features(sanity=args.sanity)
    else:
        X_flat, X_test_flat, y_raw, groups, test_fe = build_v8_features(sanity=args.sanity)
    test_groups = test_fe[cfg.GROUP_COL].values

    # 2. row → sequence 변환
    X_seq, y_seq, train_usc = flat_to_seq(X_flat, y_raw, groups)
    X_test_seq, _, test_usc = flat_to_seq(X_test_flat, None, test_groups)
    v5.log(f"  train seq: {X_seq.shape}  test seq: {X_test_seq.shape}")

    # 3. 하이퍼파라미터
    # defaults (Transformer 계열)
    model_cls, model_kwargs = StrongTransformer, {}

    if args.sanity:
        seeds, n_folds, max_epochs = [42], 2, 10
        model_kwargs = {"d_model": 64, "n_heads": 4, "n_layers": 2, "d_ff": 128, "dropout": 0.2}
        peak_lr, warmup, patience = 3e-4, 2, 10
        ckpt_prefix = "v8_transformer_sanity"
    elif args.base:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 200
        model_cls    = BiLSTMAttention
        model_kwargs = {"hidden": 256, "n_layers": 2, "n_heads": 8, "dropout": 0.2}
        peak_lr, warmup, patience = 3e-4, 10, 30
        ckpt_prefix  = "v8_base_lstm"
    elif args.lstm:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 300
        model_cls    = BiLSTMAttention
        model_kwargs = {"hidden": 256, "n_layers": 2, "n_heads": 8, "dropout": 0.2}
        peak_lr, warmup, patience = 1e-4, 20, 50
        ckpt_prefix  = "v8_lstm_attn_v2"
    elif args.cnn:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 200
        model_cls    = Conv1DNet
        model_kwargs = {"channels": 512, "dropout": 0.2}
        peak_lr, warmup, patience = 3e-4, 10, 30
        ckpt_prefix  = "v8b_cnn"
    elif args.v5:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 200
        model_kwargs = {"d_model": 128, "n_heads": 4, "n_layers": 4, "d_ff": 512, "dropout": 0.2}
        peak_lr, warmup, patience = 3e-4, 10, 30
        ckpt_prefix  = "v8b_transformer_v5"
    elif args.v4:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 250
        model_kwargs = {"d_model": 256, "n_heads": 8, "n_layers": 6, "d_ff": 1024, "dropout": 0.2}
        peak_lr, warmup, patience = 2e-4, 10, 35
        ckpt_prefix = "v8b_transformer_v4"
    elif args.v3:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 300
        model_kwargs = {"d_model": 256, "n_heads": 8, "n_layers": 6, "d_ff": 1024, "dropout": 0.3}
        peak_lr, warmup, patience = 1e-4, 20, 50
        ckpt_prefix = "v8_transformer_v3"
    elif args.v2:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 200
        model_kwargs = {"d_model": 256, "n_heads": 8, "n_layers": 6, "d_ff": 1024, "dropout": 0.2}
        peak_lr, warmup, patience = 3e-4, 10, 30
        ckpt_prefix = "v8_transformer_v2"
    else:
        seeds, n_folds, max_epochs = [42, 123, 2026], 5, 100
        model_kwargs = {"d_model": 256, "n_heads": 8, "n_layers": 4, "d_ff": 1024, "dropout": 0.2}
        peak_lr, warmup, patience = 1e-3, 0, 15
        ckpt_prefix = "v8_transformer"

    # 4. Transformer 학습
    train_loss_fn = nn.L1Loss() if (args.v4 or args.v5 or args.cnn) else None

    all_oof, all_test = [], []
    for seed in seeds:
        cv, oof_seq, test_seq = run_transformer_cv_seq(
            X_seq, y_seq, X_test_seq, train_usc,
            model_cls=model_cls, model_kwargs=model_kwargs,
            n_folds=n_folds, base_seed=seed, max_epochs=max_epochs,
            peak_lr=peak_lr, warmup=warmup, patience=patience,
            ckpt_name=f"{ckpt_prefix}_seed{seed}",
            loss_fn=train_loss_fn,
        )
        oof_flat  = seq_to_flat(oof_seq,  groups,      train_usc)
        test_flat = seq_to_flat(test_seq, test_groups, test_usc)
        all_oof.append(oof_flat); all_test.append(test_flat)
        v5.log(f"  seed={seed} CV(row): {mean_absolute_error(y_raw, oof_flat):.4f}")

    oof_tf  = np.mean(all_oof,  axis=0)
    test_tf = np.mean(all_test, axis=0)
    cv_tf   = mean_absolute_error(y_raw, oof_tf)
    v5.log(f"  {model_cls.__name__} 3-seed avg CV: {cv_tf:.4f}")

    if args.sanity:
        v5.log(f"  총 소요: {(time.time()-t0)/60:.1f}분")
        return

    # 5. N-way blend (GBDT + strict2_tabnet + v8_tabnet + transformer)
    v5.log("\n  === N-way blend ===")
    model_oofs, model_tests, model_names = [], [], []

    v5.CKPT_DIR = V8_CKPT_DIR
    for name in ["lgb_huber_log", "lgb_mae_log", "cat_mae_log", "lgb_mae_raw", "xgb_mae_raw"]:
        oofs, tests = [], []
        for seed in [42, 123, 2026]:
            ck = v5.load_ckpt(f"v8_{name}_seed{seed}")
            if ck is not None:
                _, tp, op = ck
                oofs.append(op); tests.append(tp)
        if oofs:
            model_oofs.append(np.mean(oofs, axis=0))
            model_tests.append(np.mean(tests, axis=0))
            model_names.append(name)

    v5.CKPT_DIR = strict.STRICT_CKPT_DIR
    s2_oofs, s2_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"strict_2_focused_tabnet_seed{seed}")
        if ck is not None:
            _, tp, op = ck
            s2_oofs.append(op); s2_tests.append(tp)
    if s2_oofs:
        model_oofs.append(np.mean(s2_oofs, axis=0))
        model_tests.append(np.mean(s2_tests, axis=0))
        model_names.append("strict2_tabnet")

    v5.CKPT_DIR = V8_TAB_CKPT
    t8_oofs, t8_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8_tabnet_seed{seed}")
        if ck is not None:
            _, tp, op = ck
            t8_oofs.append(op); t8_tests.append(tp)
    if t8_oofs:
        model_oofs.append(np.mean(t8_oofs, axis=0))
        model_tests.append(np.mean(t8_tests, axis=0))
        model_names.append("v8_tabnet")

    # lstm 모드: v8_transformer_v2도 포함 (9-way blend)
    v5.CKPT_DIR = V8_TF_CKPT_DIR
    if args.lstm:
        tf_oofs, tf_tests = [], []
        for seed in [42, 123, 2026]:
            ck = v5.load_ckpt(f"v8_transformer_v2_seed{seed}")
            if ck is not None:
                _, oof_c, test_c = ck
                if oof_c.shape[0] == len(X_seq):
                    tf_oofs.append(seq_to_flat(oof_c, groups, train_usc))
                    tf_tests.append(seq_to_flat(test_c, test_groups, test_usc))
        if tf_oofs:
            model_oofs.append(np.mean(tf_oofs, axis=0))
            model_tests.append(np.mean(tf_tests, axis=0))
            model_names.append("v8_transformer_v2")

    model_oofs.append(oof_tf);  model_tests.append(test_tf)
    model_names.append(ckpt_prefix)
    weights, blend_cv = v5.optimize_blend_multi(model_oofs, y_raw, model_names)

    # 6. 제출 파일
    oof_blend  = sum(w * o for w, o in zip(weights, model_oofs))
    test_blend = sum(w * t for w, t in zip(weights, model_tests))

    sub = pd.DataFrame({cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
                        cfg.TARGET: np.clip(test_blend, 0, None)})
    p_nc = os.path.join(cfg.SUBMISSION_DIR, f"{ckpt_prefix}_blend_noclip_submission.csv")
    sub.to_csv(p_nc, index=False, float_format="%.10f", lineterminator="\n")
    v5.log(f"  저장: {p_nc}  mean={sub[cfg.TARGET].mean():.3f} std={sub[cfg.TARGET].std():.3f}")

    clip_th = np.percentile(oof_blend, 99) * 1.30
    sub_c = sub.copy(); sub_c[cfg.TARGET] = np.clip(test_blend, 0, clip_th)
    p_c  = os.path.join(cfg.SUBMISSION_DIR, f"{ckpt_prefix}_blend_clip130_submission.csv")
    sub_c.to_csv(p_c, index=False, float_format="%.10f", lineterminator="\n")
    cv_clip = mean_absolute_error(y_raw, np.clip(oof_blend, 0, clip_th))
    v5.log(f"  저장: {p_c}  clip_th={clip_th:.2f} CV_clip={cv_clip:.4f}")

    v5.log("\n" + "=" * 60)
    v5.log(f"  {model_cls.__name__} CV: {cv_tf:.4f}")
    v5.log(f"  blend noclip CV:  {blend_cv:.4f}")
    v5.log(f"  blend clip130 CV: {cv_clip:.4f}")
    v5.log(f"  총 소요: {(time.time()-t0)/60:.1f}분")
    v5.log("=" * 60)


if __name__ == "__main__":
    main()


---
## run_v8_mlp.py (MLP 학습)


In [ ]:
# run_v8_mlp.py
# ============================================================
# 아래 코드를 run_v8_mlp.py 파일로 저장하세요

"""
File: run_v8_mlp.py
"""


In [ ]:
"""
v8b MLP — 798 피처 row-level TabMLP (3 seeds × 5-fold GroupKFold)
"""
import os, math, time
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

import config as cfg
import run_experiments_v5 as v5
from run_v8_tabnet import build_v8_features

BASE_DIR    = os.path.dirname(os.path.abspath(__file__))
MLP_CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v8_transformer")
V8_CKPT_DIR  = os.path.join(BASE_DIR, "checkpoints_v8")
V8_TAB_CKPT  = os.path.join(BASE_DIR, "checkpoints_v8_tabnet")
import run_experiments_strict as strict
import run_experiments_attack as attack


class TabMLP(nn.Module):
    def __init__(self, n_feat, hidden=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, hidden),   nn.BatchNorm1d(hidden),   nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden),   nn.BatchNorm1d(hidden),   nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.BatchNorm1d(hidden//2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden//2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def run_mlp_cv(X_train, y_raw, X_test, groups,
               n_folds=5, seeds=(42, 123, 2026),
               hidden=512, dropout=0.2,
               peak_lr=3e-4, warmup=10, max_epochs=200, patience=30,
               batch_size=2048, ckpt_prefix="v8b_mlp"):

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    amp     = torch.autocast("cuda", dtype=torch.bfloat16) if use_amp else nullcontext()
    n_feat  = X_train.shape[1]

    all_oof, all_test = [], []

    for seed in seeds:
        ckpt_name = f"{ckpt_prefix}_seed{seed}"
        cached = v5.load_ckpt(ckpt_name)
        if cached is not None:
            cv_c, tp_c, op_c = cached
            v5.log(f"  [캐시] seed={seed} CV={cv_c:.4f}")
            all_oof.append(op_c); all_test.append(tp_c)
            continue

        torch.manual_seed(seed)
        gkf     = GroupKFold(n_splits=n_folds)
        oof     = np.zeros(len(y_raw))
        test_acc = np.zeros(len(X_test))
        v5.log(f"  seed={seed}  device={device}")

        X_tr_t  = torch.from_numpy(X_train.astype(np.float32))
        X_te_t  = torch.from_numpy(X_test.astype(np.float32)).to(device)
        y_log   = np.log1p(y_raw).astype(np.float32)

        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, groups=groups)):
            ft0 = time.time()
            Xb = X_tr_t[tr_idx].to(device)
            yb = torch.from_numpy(y_log[tr_idx]).to(device)
            Xv = X_tr_t[val_idx].to(device)
            yv_raw = y_raw[val_idx]

            model   = TabMLP(n_feat, hidden, dropout).to(device)
            opt     = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=1e-4)
            def lr_fn(ep):
                if ep < warmup: return ep / max(warmup, 1)
                prog = (ep - warmup) / max(max_epochs - warmup, 1)
                return 0.5 * (1 + math.cos(math.pi * prog))
            sched   = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
            loss_fn = nn.L1Loss()

            best_mae, best_state, p_cnt = float("inf"), None, 0
            for epoch in range(max_epochs):
                model.train()
                perm = torch.randperm(len(Xb), device=device)
                for i in range(0, len(Xb), batch_size):
                    xb_, yb_ = Xb[perm[i:i+batch_size]], yb[perm[i:i+batch_size]]
                    with amp:
                        loss = loss_fn(model(xb_), yb_)
                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sched.step()

                if (epoch + 1) % 5 == 0:
                    model.eval()
                    with torch.no_grad(), amp:
                        vp = np.expm1(model(Xv).cpu().float().numpy()).clip(0)
                    vm = mean_absolute_error(yv_raw, vp)
                    if vm < best_mae:
                        best_mae, p_cnt = vm, 0
                        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    else:
                        p_cnt += 1
                    if p_cnt >= patience // 5:
                        v5.log(f"    early stop ep={epoch+1}  best={best_mae:.4f}")
                        break

            model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad(), amp:
                oof[val_idx]  = np.expm1(model(Xv).cpu().float().numpy()).clip(0)
                test_acc += np.expm1(model(X_te_t).cpu().float().numpy()).clip(0) / n_folds

            v5.log(f"  Fold {fold+1} MAE: {mean_absolute_error(yv_raw, oof[val_idx]):.4f}  ({time.time()-ft0:.0f}s)")

        cv = mean_absolute_error(y_raw, oof)
        v5.log(f"  seed={seed} CV: {cv:.4f}")
        v5.save_ckpt(ckpt_name, (cv, test_acc, oof))
        all_oof.append(oof); all_test.append(test_acc)

    oof_avg  = np.mean(all_oof, axis=0)
    test_avg = np.mean(all_test, axis=0)
    cv_avg   = mean_absolute_error(y_raw, oof_avg)
    v5.log(f"  MLP 3-seed avg CV: {cv_avg:.4f}")
    return cv_avg, oof_avg, test_avg


def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--sanity", action="store_true")
    args = parser.parse_args()

    v5.CKPT_DIR = MLP_CKPT_DIR
    os.makedirs(MLP_CKPT_DIR, exist_ok=True)
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    os.makedirs(cfg.LOG_DIR, exist_ok=True)

    v5.log("=" * 60)
    v5.log("v8b MLP [798 피처, row-level, L1Loss]")
    if args.sanity: v5.log("!! SANITY 모드")
    v5.log("=" * 60)
    t0 = time.time()

    X_flat, X_test_flat, y_raw, groups, test_fe = build_v8_features(sanity=args.sanity)
    seeds    = [42] if args.sanity else [42, 123, 2026]
    n_folds  = 2    if args.sanity else 5
    patience = 5    if args.sanity else 30
    max_ep   = 20   if args.sanity else 200

    cv, oof, test_pred = run_mlp_cv(
        X_flat, y_raw, X_test_flat, groups,
        n_folds=n_folds, seeds=seeds,
        peak_lr=3e-4, warmup=10, max_epochs=max_ep, patience=patience,
        batch_size=2048, ckpt_prefix="v8b_mlp",
    )

    if args.sanity:
        v5.log(f"  총 소요: {(time.time()-t0)/60:.1f}분"); return

    # N-way blend
    v5.log("\n  === N-way blend ===")
    model_oofs, model_tests, model_names = [], [], []

    v5.CKPT_DIR = V8_CKPT_DIR
    for name in ["lgb_mae_log", "lgb_huber_log", "cat_mae_log", "lgb_mae_raw", "xgb_mae_raw"]:
        oofs, tests = [], []
        for s in [42, 123, 2026]:
            ck = v5.load_ckpt(f"v8b_{name}_seed{s}") or v5.load_ckpt(f"v8_{name}_seed{s}")
            if ck: _, tp, op = ck; oofs.append(op); tests.append(tp)
        if oofs:
            model_oofs.append(np.mean(oofs, axis=0)); model_tests.append(np.mean(tests, axis=0))
            model_names.append(name)

    v5.CKPT_DIR = strict.STRICT_CKPT_DIR
    s2o, s2t = [], []
    for s in [42, 123, 2026]:
        ck = v5.load_ckpt(f"strict_2_focused_tabnet_seed{s}")
        if ck: _, tp, op = ck; s2o.append(op); s2t.append(tp)
    if s2o:
        model_oofs.append(np.mean(s2o, axis=0)); model_tests.append(np.mean(s2t, axis=0))
        model_names.append("strict2_tabnet")

    v5.CKPT_DIR = V8_TAB_CKPT
    t8o, t8t = [], []
    for s in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8_tabnet_seed{s}")
        if ck: _, tp, op = ck; t8o.append(op); t8t.append(tp)
    if t8o:
        model_oofs.append(np.mean(t8o, axis=0)); model_tests.append(np.mean(t8t, axis=0))
        model_names.append("v8_tabnet")

    v5.CKPT_DIR = MLP_CKPT_DIR
    from run_v8_transformer import flat_to_seq, seq_to_flat, V8_TF_CKPT_DIR
    v5.CKPT_DIR = V8_TF_CKPT_DIR
    test_groups = test_fe[cfg.GROUP_COL].values
    X_seq, y_seq, train_usc = flat_to_seq(X_flat, y_raw, groups)
    X_test_seq, _, test_usc = flat_to_seq(X_test_flat, None, test_groups)
    tfo, tft = [], []
    for s in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8b_transformer_v4_seed{s}") or v5.load_ckpt(f"v8_transformer_v4_seed{s}")
        if ck:
            _, oof_seq, test_seq = ck
            if oof_seq.shape[0] == len(train_usc):
                tfo.append(seq_to_flat(oof_seq, groups, train_usc))
                tft.append(seq_to_flat(test_seq, test_groups, test_usc))
    if tfo:
        model_oofs.append(np.mean(tfo, axis=0)); model_tests.append(np.mean(tft, axis=0))
        model_names.append("v8b_transformer_v4")

    model_oofs.append(oof); model_tests.append(test_pred); model_names.append("v8b_mlp")
    weights, blend_cv = v5.optimize_blend_multi(model_oofs, y_raw, model_names)

    test_blend = sum(w * t for w, t in zip(weights, model_tests))
    sub = pd.DataFrame({cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
                        cfg.TARGET: np.clip(test_blend, 0, None)})
    p = os.path.join(cfg.SUBMISSION_DIR, "v8b_mlp_blend_submission.csv")
    sub.to_csv(p, index=False, float_format="%.10f", lineterminator="\n")

    v5.log(f"\n  MLP CV:      {cv:.4f}")
    v5.log(f"  blend CV:    {blend_cv:.4f}")
    v5.log(f"  저장: {p}")
    v5.log(f"  총 소요: {(time.time()-t0)/60:.1f}분")
    v5.log("=" * 60)


if __name__ == "__main__":
    main()


---
## run_stacking.py (스태킹 메타러너)


In [ ]:
# run_stacking.py
# ============================================================
# 아래 코드를 run_stacking.py 파일로 저장하세요

"""
File: run_stacking.py
"""


In [ ]:
"""
Stacking meta-learner: OOF 8개 → LGB 메타모델
"""
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

import config as cfg
import run_experiments_v5 as v5
import run_experiments_strict as strict
from run_v8_tabnet import build_v8_features
from run_v8_transformer import flat_to_seq, seq_to_flat, V8_TF_CKPT_DIR, V8_CKPT_DIR, V8_TAB_CKPT

BASE_DIR = os.path.dirname(os.path.abspath(__file__))


def collect_oofs(groups, test_groups, train_usc, test_usc):
    model_oofs, model_tests, model_names = [], [], []

    # GBDT — v8b(시나리오 피처 추가) 있으면 우선 사용, 없으면 v8 폴백
    v5.CKPT_DIR = V8_CKPT_DIR
    for name in ["lgb_mae_log", "lgb_huber_log", "cat_mae_log", "lgb_mae_raw", "xgb_mae_raw"]:
        oofs, tests = [], []
        for seed in [42, 123, 2026]:
            ck = v5.load_ckpt(f"v8b_{name}_seed{seed}") or v5.load_ckpt(f"v8_{name}_seed{seed}")
            if ck is not None:
                _, tp, op = ck
                oofs.append(op); tests.append(tp)
        if oofs:
            model_oofs.append(np.mean(oofs, axis=0))
            model_tests.append(np.mean(tests, axis=0))
            model_names.append(name)

    # strict2 TabNet
    v5.CKPT_DIR = strict.STRICT_CKPT_DIR
    s2_oofs, s2_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"strict_2_focused_tabnet_seed{seed}")
        if ck is not None:
            _, tp, op = ck
            s2_oofs.append(op); s2_tests.append(tp)
    if s2_oofs:
        model_oofs.append(np.mean(s2_oofs, axis=0))
        model_tests.append(np.mean(s2_tests, axis=0))
        model_names.append("strict2_tabnet")

    # v8 TabNet
    v5.CKPT_DIR = V8_TAB_CKPT
    t8_oofs, t8_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8_tabnet_seed{seed}")
        if ck is not None:
            _, tp, op = ck
            t8_oofs.append(op); t8_tests.append(tp)
    if t8_oofs:
        model_oofs.append(np.mean(t8_oofs, axis=0))
        model_tests.append(np.mean(t8_tests, axis=0))
        model_names.append("v8_tabnet")

    # Transformer v4 — v8b(798 피처) 있으면 우선, 없으면 v8 폴백
    v5.CKPT_DIR = V8_TF_CKPT_DIR
    tf_oofs, tf_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8b_transformer_v4_seed{seed}") or v5.load_ckpt(f"v8_transformer_v4_seed{seed}")
        if ck is not None:
            _, oof_seq, test_seq = ck
            if oof_seq.shape[0] == len(train_usc):
                tf_oofs.append(seq_to_flat(oof_seq, groups, train_usc))
                tf_tests.append(seq_to_flat(test_seq, test_groups, test_usc))
    if tf_oofs:
        model_oofs.append(np.mean(tf_oofs, axis=0))
        model_tests.append(np.mean(tf_tests, axis=0))
        model_names.append("v8_transformer_v4")

    # MLP
    mlp_oofs, mlp_tests = [], []
    for seed in [42, 123, 2026]:
        ck = v5.load_ckpt(f"v8b_mlp_seed{seed}")
        if ck is not None:
            _, tp, op = ck
            mlp_oofs.append(op); mlp_tests.append(tp)
    if mlp_oofs:
        model_oofs.append(np.mean(mlp_oofs, axis=0))
        model_tests.append(np.mean(mlp_tests, axis=0))
        model_names.append("v8b_mlp")

    # CNN + v5 transformer (sequence → flat)
    for ckpt_key, label in [("v8b_cnn", "v8b_cnn"), ("v8b_transformer_v5", "v8b_tf_v5")]:
        seq_oofs, seq_tests = [], []
        for seed in [42, 123, 2026]:
            ck = v5.load_ckpt(f"{ckpt_key}_seed{seed}")
            if ck is not None:
                _, oof_seq, test_seq = ck
                if oof_seq.shape[0] == len(train_usc):
                    seq_oofs.append(seq_to_flat(oof_seq, groups, train_usc))
                    seq_tests.append(seq_to_flat(test_seq, test_groups, test_usc))
        if seq_oofs:
            model_oofs.append(np.mean(seq_oofs, axis=0))
            model_tests.append(np.mean(seq_tests, axis=0))
            model_names.append(label)

    return model_oofs, model_tests, model_names


def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--tag", default="v1", help="제출 파일 버전 태그 (예: v1, v2)")
    parser.add_argument("--method", default="lgb", choices=["lgb", "ridge", "blend"], help="메타러너 종류")
    args = parser.parse_args()

    v5.log("=" * 60)
    v5.log(f"Stacking meta-learner [tag={args.tag}, method={args.method}]")
    v5.log("=" * 60)

    # 1. 피처/그룹 로드 (OOF 정렬용)
    X_flat, X_test_flat, y_raw, groups, test_fe = build_v8_features()
    test_groups = test_fe[cfg.GROUP_COL].values
    X_seq, y_seq, train_usc = flat_to_seq(X_flat, y_raw, groups)
    X_test_seq, _, test_usc = flat_to_seq(X_test_flat, None, test_groups)

    # 2. OOF 수집
    model_oofs, model_tests, model_names = collect_oofs(groups, test_groups, train_usc, test_usc)
    v5.log(f"수집된 모델 ({len(model_names)}개): {model_names}")

    # 블렌드 CV 기준선
    weights, blend_cv = v5.optimize_blend_multi(model_oofs, y_raw, model_names)
    v5.log(f"Blend CV (기준선): {blend_cv:.4f}")

    # 3. 메타 피처 구성
    X_meta      = np.column_stack(model_oofs)
    X_test_meta = np.column_stack(model_tests)

    # 4. 메타러너 (method에 따라 분기)
    gkf      = GroupKFold(n_splits=5)
    oof_meta = np.zeros(len(y_raw))
    test_meta = np.zeros(len(test_groups))

    if args.method == "blend":
        oof_meta  = sum(w * o for w, o in zip(weights, model_oofs))
        test_meta = sum(w * t for w, t in zip(weights, model_tests))

    elif args.method == "ridge":
        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_meta, groups=groups)):
            sc = StandardScaler()
            X_tr = sc.fit_transform(np.log1p(X_meta[tr_idx]))
            X_val = sc.transform(np.log1p(X_meta[val_idx]))
            y_tr = np.log1p(y_raw[tr_idx])

            model = Ridge(alpha=10.0)
            model.fit(X_tr, y_tr)

            oof_meta[val_idx] = np.expm1(model.predict(X_val)).clip(0)
            test_meta += np.expm1(model.predict(sc.transform(np.log1p(X_test_meta)))).clip(0) / gkf.n_splits

            fold_mae = mean_absolute_error(y_raw[val_idx], oof_meta[val_idx])
            v5.log(f"  Fold {fold+1} MAE: {fold_mae:.4f}")

    else:  # lgb
        params = {
            "objective": "mae", "metric": "mae",
            "num_leaves": 15, "learning_rate": 0.05,
            "min_child_samples": 100, "subsample": 0.8,
            "colsample_bytree": 1.0, "verbose": -1, "seed": 42,
        }
        for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_meta, groups=groups)):
            X_tr, y_tr = X_meta[tr_idx], np.log1p(y_raw[tr_idx])
            X_val, y_val = X_meta[val_idx], np.log1p(y_raw[val_idx])
            dtrain = lgb.Dataset(X_tr, label=y_tr)
            dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            model = lgb.train(params, dtrain, num_boost_round=3000, valid_sets=[dval],
                              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
            oof_meta[val_idx] = np.expm1(model.predict(X_meta[val_idx]))
            test_meta += np.expm1(model.predict(X_test_meta)) / gkf.n_splits
            fold_mae = mean_absolute_error(y_raw[val_idx], oof_meta[val_idx])
            v5.log(f"  Fold {fold+1} MAE: {fold_mae:.4f}  (best_iter={model.best_iteration})")

    meta_cv = mean_absolute_error(y_raw, oof_meta)
    v5.log(f"\n{args.method} CV: {meta_cv:.4f}")
    v5.log(f"Blend CV (기준): {blend_cv:.4f}")
    v5.log(f"개선:             {blend_cv - meta_cv:+.4f}")

    # 5. 제출 파일 저장
    os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
    sub = pd.DataFrame({
        cfg.ID_COL: test_fe[cfg.ID_COL].astype(str),
        cfg.TARGET: np.clip(test_meta, 0, None),
    })
    p = os.path.join(cfg.SUBMISSION_DIR, f"v8_stacking_{args.tag}_submission.csv")
    sub.to_csv(p, index=False, float_format="%.10f", lineterminator="\n")
    v5.log(f"저장: {p}  mean={sub[cfg.TARGET].mean():.3f}  std={sub[cfg.TARGET].std():.3f}")
    v5.log("=" * 60)


if __name__ == "__main__":
    main()
